# 21. Colab 27B 본 실행

담당: 준혁

이 노트북은 위에서 아래로 한 번만 읽고 실행하도록 만들었습니다. 맨 위의 파라미터 셀에서
모든 선택을 끝내고, 그 아래의 셀들은 순서대로 실행하기만 하면 됩니다. 오래 걸리는 작업은
모두 `python -m vqa.jobs` 로 분리해서 띄우므로, 셀을 멈추거나 브라우저를 닫아도 계속
돌아갑니다.

## 이 노트북이 수행하는 순서

1. 파라미터를 정하고 Drive를 연결한 다음 코드를 받습니다.
2. 환경을 설치하고 확인합니다. 가속 커널을 쓸 수 있는지도 여기에서 봅니다.
3. 현재 GPU의 VRAM을 재서 정밀도와 run-id를 확정하고, 가중치를 캐시에 확보합니다.
4. 데이터를 로컬 디스크에 풉니다.
5. preflight로 학습 전 점검을 수행합니다.
6. 학습하지 않은 원본 27B로 fold 0을 추론하고 평가합니다.
7. (선택) 다른 프롬프트로, 또는 해상도를 올린 설정으로 같은 fold 0을 추론해서 비교합니다.
8. 원본 27B로 전체 test를 추론하고 제출 파일을 만듭니다.
9. 학습을 시작합니다. 학습은 열 시간 안팎이 걸리므로 기다리지 않고 넘어갑니다.
10. 학습이 끝난 뒤에 fold 0을 다시 추론하고, 원본과 문항 단위로 비교합니다. 전체 데이터로
    학습했다면 이 단계는 건너뜁니다.
11. 학습된 모델로 전체 test를 추론하고 제출 파일을 만듭니다.

## GPU 작업은 한 번에 하나만 돌립니다

2026-09-21에 GPU 셀 네 개를 연달아 실행해서 모델 네 개가 동시에 올라갔고, 그 결과로
out of memory가 났습니다. `vqa.jobs start` 는 job을 띄우자마자 곧바로 돌아오기 때문입니다.
그래서 이 노트북의 GPU 단계는 job을 띄운 다음 `vqa.jobs wait` 로 끝날 때까지 기다립니다.
`vqa.jobs start` 자체도 같은 run-root에 살아 있는 job이 있으면 시작을 거부합니다. 메뉴의
"모두 실행"을 눌러도 작업이 겹치지 않습니다.

예외는 학습 하나뿐입니다. 학습은 열 시간 안팎이 걸려서 셀이 그만큼 붙잡혀 있으면 곤란하므로,
시작만 하고 기다리지 않습니다. 대신 학습 다음에 오는 15번 절부터 17번 절까지의 셀은 실행하기
전에 학습 job이 아직 살아 있는지와 `final_adapter` 가 만들어졌는지를 스스로 확인하고, 아직
때가 아니면 안내만 출력하고 아무 일도 하지 않습니다. 그러므로 학습을 띄운 직후에 "모두 실행"이
계속 내려가더라도 GPU를 함께 쓰는 일은 생기지 않습니다. 학습이 끝난 다음에 그 셀들을 차례로
다시 실행하시면 됩니다.

## 이미 끝난 작업은 건너뜁니다

`vqa.predict` 는 예측 파일에 이미 들어 있는 문항을 건너뛰고 남은 문항만 추론합니다. 예측이
완전히 끝나면 파일 옆에 `.manifest.json` 이 생기고, 이 노트북의 도우미 함수는 그 파일이
있으면 모델을 다시 올리지 않고 해당 단계를 건너뜁니다. `vqa.train` 은 `--resume auto` 가
기본이므로 Drive에 남은 마지막 checkpoint에서 이어집니다. 그래서 어느 셀이든 다시 실행해도
안전합니다.

## 세션이 끊긴 뒤에 다시 시작하는 방법

가상 머신은 언제든 회수될 수 있습니다. 다시 접속했다면 아래 세 가지만 다시 실행하면 됩니다.

1. 파라미터 셀
2. Drive 연결 셀과 코드 가져오기 셀, 부트스트랩 셀
3. 환경 설치와 확인 셀, GPU와 정밀도 결정 셀, 데이터 준비 셀
4. 실행 도우미 셀 (7번 절)

여기까지 실행한 다음에는 중단된 지점의 셀을 그대로 다시 실행하십시오. 명령을 고칠 필요가
없습니다. 예측은 남은 문항부터, 학습은 마지막 checkpoint부터 이어집니다. 상황별 대응은
`docs/colab_survival.md` 에 더 자세히 적어 두었습니다.

## 1. 파라미터

이 셀이 이 노트북의 유일한 선택 지점입니다. 아래에서 다시 묻거나 다시 정하는 곳은 없습니다.

`PROMPT_ID` 를 바꾸면 run-id에 프롬프트 이름이 접미사로 붙습니다. 기본값인 `v1_image_only`
일 때에만 접미사가 붙지 않고 기존 이름을 그대로 씁니다. 프롬프트가 다른 결과가 같은 run 폴더에
섞이는 일을 막기 위해서입니다. 학습과 추론에 쓰는 모든 명령이 같은 `PROMPT_ID` 를
`--set prompt_id=...` 로 받으므로, 학습할 때와 추론할 때의 프롬프트는 항상 같습니다.

`CONFIG` 를 비워 두면 아래 셀이 정밀도 정책과 VRAM을 보고 설정 파일을 정합니다. 작은 글자를
더 읽히고 싶다면 `CONFIG = UPSCALE_CONFIG` 라고 적으십시오. 확대본은 `min_pixels` 를 올려서
작은 이미지를 키우는 설정이며, run-id 뒤에 `_upscale` 이 붙어서 기존 결과와 섞이지 않습니다.
상한이 아니라 하한을 올리는 까닭은, 작은 글자 때문에 틀린 fold 0의 12문항이 모두 720×960
정도여서 기존 상한에 걸리지도 않았기 때문입니다. 대신 train 전체의 평균 시각 토큰이 702개에서
1,027개로 약 1.46배가 되므로 VRAM과 학습 시간이 함께 올라갑니다. 8번 절의 preflight로
`peak_vram_gb` 와 `sec_per_optimizer_step` 을 먼저 확인한 다음에 학습을 시작하십시오.

모델을 Qwen3.8-27B로 바꾸어 비교하려면 `CONFIG = QWEN38_UPSCALE_CONFIG` 라고 적으십시오.
해상도와 LoRA 설정과 학습 하이퍼파라미터는 위의 확대본과 같고 모델만 다르므로, 두 run의
차이를 모델 하나로 설명할 수 있습니다. 모델 식별자와 revision은 5번 절이 이 설정 파일에서
직접 읽으므로 파라미터 셀의 `MODEL_ID` 를 따로 고칠 필요가 없습니다. 절차와 예상 시간은
`docs/colab_personal_account.md` 의 Qwen3.8-27B 비교 실험 절에 정리해 두었습니다.

`TRAIN_SCOPE` 는 학습에 쓰는 문항의 범위를 정합니다. 기본값인 `"fold0"` 은 fold 1~4의
5,358문항만 학습하고 fold 0의 1,344문항을 검증으로 남겨 둡니다. 설정을 비교하고 학습 효과를
판단하는 단계에서 쓰는 값입니다. `"full"` 로 바꾸면 fold 0까지 합친 6,690문항을 모두 학습하며,
최종 제출 모델을 만들 때에만 씁니다. `"full"` 일 때에는 학습 run-id 뒤에 `_full` 이 붙어서
기존 run과 섞이지 않고, 15번 절과 16번 절의 fold 0 평가는 안내만 출력하고 건너뜁니다. 전체
데이터로 학습한 모델은 fold 0 문항도 학습에 썼으므로, 그 정확도가 일반화 성능을 나타내지 않기
때문입니다. 17번 절의 test 추론과 제출 파일 생성은 두 경우 모두 그대로 동작합니다.

`MODEL_CACHE` 는 27B 가중치를 어디에 둘지 정합니다. 기본값인 `"auto"` 는 20번 노트북이 남긴
캐시를 먼저 찾고, 없으면 Drive 여유가 `MODEL_CACHE_NEED_GB` 이상일 때에만 Drive를 씁니다.
개인 계정처럼 Drive 용량이 부족한 상황이라면 `"local"` 로 적어서 가상 머신의 로컬 디스크에
받으십시오. 내려받는 파일은 원본 bf16 스냅샷이며 약 54GB입니다. `nf4` 로 돌리더라도 양자화는
가중치를 GPU에 올리는 시점에 수행하므로 캐시에 필요한 용량은 줄지 않습니다. 자세한 절차는
`docs/colab_personal_account.md` 에 정리해 두었습니다.

`RUN_RESOLUTION_COMPARE` 를 `True` 로 바꾸면 10-1절이 `RESOLUTION_COMPARE_CONFIG` 로 fold 0을
한 번 더 추론해서 해상도만 다른 짝을 만듭니다. 프롬프트와 정밀도는 9번 절과 같게 둡니다.

`ALSO_BASE` 는 학습 모델로 추론할 때 원본 모델의 확률까지 같은 forward에서 계산할지를
정합니다. 기본값은 `False` 입니다. 같은 프롬프트의 원본 예측 파일을 9번 절과 11번 절에서 이미
만들어 두었다면, 같은 값을 다시 계산하는 셈이라 시간만 더 듭니다. 2026-09-21 실행에서 원본
test 추론이 43분이었는데 학습 모델의 test 추론이 97분이 걸린 까닭이 이것입니다.

`ADAPTER_RUN_ID` 는 추론에 쓸 adapter를 다른 run에서 빌려 오고 싶을 때에만 적습니다. 비워
두면 지금까지와 똑같이 이 노트북이 학습한 `FT_RUN` 의 `final_adapter` 를 씁니다. run-id를 적으면
15번 절과 17번 절의 추론 명령이 `VQA_RUN_ROOT/<ADAPTER_RUN_ID>/final_adapter` 를 `--adapter` 로
넘기고, 결과는 그대로 지금 `FT_RUN` 폴더 아래에 저장하되 파일 이름 뒤에 `_xadapter` 를
붙입니다. 예를 들어 fold 0 예측은 `fold0_ft_xadapter.csv`, test 예측은
`test_ft_xadapter.csv`, 제출 파일은 `artifacts/submission/<FT_RUN>_ft_xadapter.csv` 가 됩니다.
16번 절의 비교 결과도 같은 접미사를 따라갑니다. 지정한 run에 `final_adapter` 가 없으면 5번
절에서 곧바로 오류를 내고 멈춥니다.

이 값을 쓰는 상황은 다음과 같습니다. 기본 해상도로 학습해 둔 adapter
(`colab27b_ft_bf16_v2_read_text`)를 그대로 두고, 추론만 확대 설정(`CONFIG = UPSCALE_CONFIG`)
으로 다시 돌려서 작은 글자 문항이 실제로 좋아지는지 확인하는 경우입니다. 학습을 다시 하지
않으므로 A100 시간을 크게 아낄 수 있습니다. 다만 주의할 점이 하나 있습니다. 이렇게 하면
adapter가 학습된 해상도와 추론에 쓰는 해상도가 서로 달라집니다. LoRA 가중치는 학습할 때 본
시각 토큰 수에 맞추어 조정된 값이므로, 해상도를 바꾸어 추론하면 순수한 해상도 효과만
측정되지 않고 학습 조건과 추론 조건이 어긋난 효과까지 섞여 들어옵니다. 따라서 이 결과는
빠른 예비 확인으로만 쓰시고, 확대 설정이 실제로 도움이 된다는 결론을 내리려면 확대 설정으로
처음부터 학습한 run과 비교하셔야 합니다.

In [1]:
# ===== 파라미터 (이 노트북의 모든 선택을 여기에서 끝냅니다) =====

# --- 경로 ---
PROJECT_ROOT = "/content/ssafy-16-2-ai"
REPO_URL = "https://github.com/bnbong/Astralopithecus.git"   # 비공개 저장소입니다.
REPO_BRANCH = "main"
CODE_ZIP = ""                                     # 토큰을 쓸 수 없을 때 사용할 Drive의 코드 zip 경로
DRIVE_MOUNT = "/content/drive"
DRIVE_DIR = "/content/drive/MyDrive/vqa"
ARCHIVE_PATH = "/content/drive/MyDrive/ssafy-16-2-ai.zip"
ARCHIVE_PASSWORD = ""                             # 비워 두면 실행할 때 입력받습니다. 암호가 없으면 "none"
LOCAL_SCRATCH = "/content/scratch"
DATA_ROOT = "/content/data"
RUN_ROOT = "/content/drive/MyDrive/vqa/runs"
MODEL_ROOT = "/content/drive/MyDrive/vqa/hf_cache"   # MODEL_CACHE 가 "drive" 이거나 "auto" 일 때 쓰는 Drive 캐시입니다.
LOCAL_MODEL_ROOT = "/content/hf_cache"               # 가상 머신 로컬 디스크의 캐시입니다.

# 가중치 캐시를 어디에 둘지 정합니다.
#   "auto"  : 20번 노트북이 남긴 캐시가 있으면 그것을 쓰고, 없으면 Drive 여유를 보고 고릅니다.
#   "drive" : 언제나 MODEL_ROOT 를 씁니다. 가상 머신이 바뀌어도 다시 받지 않습니다.
#   "local" : 언제나 LOCAL_MODEL_ROOT 를 씁니다. Drive 용량이 부족한 개인 계정에서 쓰십시오.
#             런타임이 회수되면 가중치가 사라지므로 접속할 때마다 다시 받아야 합니다.
MODEL_CACHE = "auto"
MODEL_CACHE_NEED_GB = 60.0        # 27B 원본 스냅샷은 약 54GB입니다. 여유를 포함한 값입니다.
# Qwen3.5와 Qwen3.8을 번갈아 돌리면 캐시에 54GB짜리 스냅샷이 두 벌, 모두 약 108GB가 쌓입니다.
# 한 벌만 두고 싶다면 다 쓴 models--Qwen--Qwen3.5-27B 폴더를 직접 지우십시오.

# --- 모델과 프롬프트 ---
# 아래 두 값은 5번 절에서 확정된 CONFIG 의 model_id 와 revision 으로 덮어씁니다. 여기 적힌
# 값은 CONFIG 를 읽기 전에 화면에 무엇을 기대하고 있는지 보여 주는 용도입니다. 모델을 바꾸려면
# 이 두 줄이 아니라 아래의 CONFIG 를 바꾸십시오.
MODEL_ID = "Qwen/Qwen3.5-27B"
MODEL_REVISION = "fc05daec18b0a78c049392ed2e771dde82bdf654"
# 캐시에 스냅샷이 없으면 5-1절에서 먼저 내려받습니다. 추론 job이 가중치를 받는 도중에
# 재시작되는 상황을 피하기 위해서입니다.
DOWNLOAD_WEIGHTS = True
# 정밀도 정책입니다. 팀 결정은 bf16입니다. "auto"는 VRAM만 보고 고르고, "nf4"는 항상 nf4를 씁니다.
PRECISION_POLICY = "bf16"
BF16_CONFIG = "configs/colab_qwen35_27b_bf16.yaml"
NF4_CONFIG = "configs/colab_qwen35_27b_nf4.yaml"
UPSCALE_CONFIG = "configs/colab_qwen35_27b_bf16_upscale.yaml"   # 작은 이미지 확대본 (min_pixels 상향)
# Qwen3.8-27B 비교 실험용 설정입니다. 해상도와 학습 설정은 위의 확대본과 같고 모델만 다릅니다.
QWEN38_UPSCALE_CONFIG = "configs/colab_qwen38_27b_bf16_upscale.yaml"
CONFIG = UPSCALE_CONFIG                       # 비워 두면 아래 셀이 정책과 VRAM을 보고 정합니다. 직접 적으면 그 값을 씁니다.
PROMPT_ID = "v2_read_text"       # 학습과 추론이 모두 이 프롬프트를 씁니다.

# --- run-id의 기본 이름 ---
# 실제 run-id는 여기에 프롬프트 접미사와 정밀도 접미사를 붙여서 만듭니다. 앞머리의 "colab27b"
# 는 Qwen3.5-27B를 가리키는 이름이며, CONFIG 의 model_id 가 다른 모델이면 5번 절이 그 부분만
# 바꿉니다. 예를 들어 Qwen3.8-27B이면 "colab38_27b_base_bf16" 이 됩니다.
BASE_RUN_ID = "colab27b_base_bf16"
FT_RUN_ID = "colab27b_ft_bf16"

# --- 데이터 분할 ---
# TRAIN_SCOPE 가 "fold0" 이면 fold 1~4만 학습하고 fold 0으로 검증합니다. 설정을 비교하는
# 단계에서 쓰는 값이며 기본값입니다. "full" 로 바꾸면 fold 0까지 합친 6,690문항을 모두
# 학습합니다. 최종 제출 모델을 만들 때에만 쓰십시오.
TRAIN_SCOPE = "full"
SPLIT_FILE_FOLD0 = "artifacts/data/split_fold0.json"  # 팀 공식 학습 분할 (train_fold1234, 5,358문항)
SPLIT_FILE_FULL = "artifacts/data/split_full.json"    # fold 0까지 합친 전체 (6,690문항)
VAL_IDS_FILE = "fold_csv_files/val_fold0.csv"      # 팀 공식 검증 분할 (fold 0, 1,344문항)
VAL_OUT_BASE = "fold0_base"
VAL_OUT_FT = "fold0_ft"
TEST_OUT_BASE = "test_base"
TEST_OUT_FT = "test_ft"
N_TEST_ITEMS = 6714

# --- 선택 단계: 다른 run의 adapter로 추론 ---
# 비워 두면 이 노트북이 학습한 FT_RUN의 adapter를 그대로 씁니다. run-id를 적으면 그 run의
# final_adapter를 --adapter 로 넘겨서 추론만 다시 합니다. 결과는 지금 FT_RUN 아래에 저장되며
# 파일 이름 뒤에 "_xadapter" 가 붙습니다. 기본 해상도로 학습한 adapter를 확대 설정으로
# 추론할 때 쓰는 값입니다.
ADAPTER_RUN_ID = ""

# --- 선택 단계: 프롬프트 비교 ---
RUN_PROMPT_COMPARE = False        # True로 바꾸면 아래 프롬프트로 fold 0을 한 번 더 추론합니다.
COMPARE_PROMPT_ID = "v2_read_text"

# --- 선택 단계: 해상도 비교 ---
# True로 바꾸면 10-1절이 같은 프롬프트, 같은 정밀도로 fold 0을 한 번 더 추론합니다. 이때에만
# 아래의 확대본 설정을 쓰므로, 9번 절의 결과와 비교하면 해상도 하나만 달라진 짝을 얻습니다.
RUN_RESOLUTION_COMPARE = False
RESOLUTION_COMPARE_CONFIG = UPSCALE_CONFIG
# 작은 글자 때문에 틀렸다고 판정한 문항 목록입니다. 10-1절이 이 문항들의 정오 변화를 따로
# 표로 출력합니다. 파일이 없으면 그 표만 건너뜁니다.
ERROR_TYPES_CSV = "artifacts/data/fold0_error_types_27b.csv"
SMALL_TEXT_CONSENSUS = "모델 오류: 작은 글자"

# --- preflight ---
PREFLIGHT_TRAIN_STEPS = 8
PREFLIGHT_INFER_ITEMS = 20
# preflight의 OOM 표본에 원본 픽셀 수가 가장 큰 이미지를 몇 장 강제로 넣을지 정합니다.
# 이 값이 0이면 OOM 표본을 글자 길이와 무작위로만 고르므로, 해상도를 올린 설정에서 가장 큰
# 이미지가 표본에 빠지면 preflight가 OOM을 놓칩니다.
PREFLIGHT_LARGEST_IMAGES = 2

# --- job 실행 ---
AUTO_RESTART = 3                  # 비정상 종료 시 다시 띄울 횟수입니다. 1 이상이어야 합니다.
# ALSO_BASE 를 True로 두면 학습 모델 추론이 같은 forward에서 원본 확률도 함께 계산해
# `*.base.csv` 로 남깁니다. 같은 프롬프트의 원본 예측이 이미 있으면 같은 값을 다시 계산하는
# 셈이라 시간만 듭니다. 2026-09-21 test 추론이 원본 43분, 학습 모델 97분이었던 까닭이
# 이것입니다. 결합 후보를 새로 계산해야 할 때에만 True로 바꾸십시오.
ALSO_BASE = False
JOB_POLL_SEC = 30                 # 기다리는 셀이 상태를 확인하는 간격입니다.
MIN_FREE_GB = 0                   # GPU 여유 메모리 하한입니다. 0이면 정밀도에 맞춰 자동으로 정합니다.

## 2. Drive 연결

checkpoint와 예측 파일은 모두 Drive에 남습니다. 가상 머신이 회수되어도 Drive의 내용은
그대로이므로, 다시 접속한 다음에 이어서 실행할 수 있습니다.

In [2]:
from google.colab import drive
drive.mount(DRIVE_MOUNT)
print("Drive를 연결했습니다.")

Mounted at /content/drive
Drive를 연결했습니다.


## 3. 코드 가져오기와 부트스트랩

가상 머신이 회수되면 `/content` 아래가 전부 사라지므로 코드도 함께 사라집니다. 부트스트랩
셀이 `PROJECT_ROOT` 로 작업 디렉터리를 옮기기 전에, 아래 셀이 코드를 먼저 확보합니다. 인증을
준비하는 방법은 `20_colab_27b_setup_preflight.ipynb` 의 1-1절에 정리해 두었습니다. 요약하면
이 저장소에만 Contents 읽기 권한을 준 fine-grained personal access token을 Colab 비밀
`GITHUB_TOKEN` 으로 등록하거나, 코드 zip을 Drive에 올리고 `CODE_ZIP` 에 그 경로를 적으면
됩니다. 셀은 토큰을 화면과 로그 어디에도 남기지 않습니다.

이미 돌고 있는 job은 이미 읽어 들인 코드로 계속 실행되므로 지금 `git pull` 을 해도 영향을
받지 않습니다. 다만 코드나 config가 바뀐 상태에서 job을 다시 띄우면 `--resume auto` 가 config
해시 불일치를 이유로 재개를 거부할 수 있습니다. 그러므로 코드 갱신은 실행 중인 작업이 없는
사이에만 하십시오.

In [3]:
# --- 코드 가져오기 (부트스트랩보다 먼저 실행합니다. BOOT의 헬퍼를 쓰지 않습니다.) ---
import os, re, shutil, subprocess, zipfile
from pathlib import Path

_code_root = Path(PROJECT_ROOT).expanduser()


def scrub_secret(text, secret=None):
    """출력에 토큰이 섞여 나가지 않도록 가립니다."""
    if not text:
        return ""
    if secret:
        text = text.replace(secret, "***")
    return re.sub(r"(https://)[^/@\s]*@", r"\1***@", text)


def _git(args, secret=None, check=True):
    """git 명령을 조용히 실행하고, 출력에서 토큰을 지운 뒤 돌려줍니다."""
    proc = subprocess.run(
        ["git"] + [str(a) for a in args],
        stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
        text=True, encoding="utf-8", errors="replace",
    )
    out = scrub_secret(proc.stdout, secret)
    if check and proc.returncode != 0:
        raise RuntimeError("git 명령이 실패했습니다 (rc=%d).\n%s" % (proc.returncode, out.strip()))
    return proc.returncode, out


def _read_github_token():
    """Colab 비밀 저장소에서 토큰을 읽습니다. 없으면 None을 돌려줍니다."""
    try:
        from google.colab import userdata
        return userdata.get("GITHUB_TOKEN") or None
    except Exception:
        # SecretNotFoundError, NotebookAccessError, ImportError를 모두 여기에서 받습니다.
        return None


def _flatten_single_top_dir(target):
    """압축 안에 최상위 폴더가 하나뿐이면 그 안의 내용을 target 바로 아래로 올립니다."""
    entries = [p for p in target.iterdir() if p.name != "__MACOSX"]
    if len(entries) == 1 and entries[0].is_dir():
        inner = entries[0]
        for item in list(inner.iterdir()):
            shutil.move(str(item), str(target / item.name))
        inner.rmdir()
        return True
    return False


CODE_VERSION = None

if (_code_root / ".git").exists():
    rc, out = _git(["-C", str(_code_root), "pull", "--ff-only"], check=False)
    if rc == 0:
        print("코드를 최신 상태로 갱신했습니다.")
    else:
        print("git pull이 실패했습니다. 이미 받아 둔 코드를 그대로 사용하고 계속 진행합니다.")
        print(out.strip()[:600])
    _, _head = _git(["-C", str(_code_root), "rev-parse", "HEAD"], check=False)
    CODE_VERSION = _head.strip()
else:
    _token = _read_github_token()
    if _token:
        print("Colab 비밀 GITHUB_TOKEN을 찾았습니다. 저장소를 복제합니다.")
        _code_root.parent.mkdir(parents=True, exist_ok=True)
        _auth_url = REPO_URL.replace("https://", "https://x-access-token:%s@" % _token, 1)
        _rc, _out = _git(["clone", "--depth", "1", "--branch", REPO_BRANCH,
                          _auth_url, str(_code_root)], secret=_token, check=False)
        if _rc != 0:
            raise RuntimeError("저장소 복제에 실패했습니다. 토큰 권한과 저장소 주소를 확인하십시오.\n"
                               + _out.strip()[:600])
        # 토큰이 .git/config에 남지 않도록 곧바로 원래 주소로 되돌립니다.
        _git(["-C", str(_code_root), "remote", "set-url", "origin", REPO_URL], secret=_token)
        _, _head = _git(["-C", str(_code_root), "rev-parse", "HEAD"], check=False)
        CODE_VERSION = _head.strip()
        del _auth_url, _token
    elif CODE_ZIP and Path(CODE_ZIP).expanduser().exists():
        _zip_path = Path(CODE_ZIP).expanduser()
        print("토큰이 없어 Drive의 코드 압축 파일을 사용합니다:", _zip_path)
        _code_root.mkdir(parents=True, exist_ok=True)
        with zipfile.ZipFile(_zip_path) as _zf:
            _zf.extractall(_code_root)
        if _flatten_single_top_dir(_code_root):
            print("압축 안의 최상위 폴더 하나를 펼쳐서 %s 바로 아래로 옮겼습니다." % _code_root)
        CODE_VERSION = "zip:" + _zip_path.name
    else:
        raise RuntimeError(
            "코드를 가져올 방법이 없습니다. 이 저장소는 비공개이므로 다음 두 가지 중 하나를 준비해야 합니다.\n"
            "1) GitHub에서 이 저장소에만 Contents 읽기 권한을 준 fine-grained personal access token을 만든 다음, "
            "Colab 왼쪽의 열쇠 아이콘 패널에서 이름을 GITHUB_TOKEN으로 등록하고 현재 노트북의 접근을 허용하십시오.\n"
            "2) 코드 전체를 zip으로 묶어 Drive에 올린 다음, 파라미터 셀의 CODE_ZIP에 그 경로를 적으십시오."
        )

if not (_code_root / "vqa").is_dir():
    print("주의: %s 아래에 vqa 패키지가 보이지 않습니다. 압축 파일의 구조를 확인하십시오." % _code_root)

print("코드 위치 :", _code_root)
print("코드 버전 :", CODE_VERSION)

Colab 비밀 GITHUB_TOKEN을 찾았습니다. 저장소를 복제합니다.
코드 위치 : /content/ssafy-16-2-ai
코드 버전 : adc3727407a0984c44ea0409b8c44ba72beee075


In [4]:
# --- 공통 부트스트랩: 경로, 환경 변수, 명령 실행 헬퍼 (수정할 일 거의 없음) ---
import os, sys, json, shlex, subprocess, time, csv, hashlib, random
from pathlib import Path

PROJECT_ROOT = Path(PROJECT_ROOT).expanduser().resolve()
os.chdir(PROJECT_ROOT)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))


def _setenv(name, value, create=False):
    if not value:
        return None
    p = Path(value).expanduser()
    if create:
        p.mkdir(parents=True, exist_ok=True)
    os.environ[name] = str(p)
    return p


DATA_ROOT_P = _setenv("VQA_DATA_ROOT", DATA_ROOT)
RUN_ROOT_P = _setenv("VQA_RUN_ROOT", RUN_ROOT, create=True)
SCRATCH_P = _setenv("VQA_LOCAL_SCRATCH", LOCAL_SCRATCH, create=True)
MODEL_ROOT_P = _setenv("VQA_MODEL_ROOT", MODEL_ROOT)

PY = sys.executable


def run(cmd, check=True, echo=True, quiet=False, hint=True):
    """명령을 실행하면서 출력을 그대로 보여 주고 종료 코드와 소요 시간을 돌려줍니다.

    quiet가 True이면 명령의 출력도, 뒤에 붙는 "[rc=...]" 줄도 찍지 않습니다. 상태를 조회하는
    호출은 실패가 정상적인 결과입니다. 예를 들어 `vqa.jobs status` 는 죽은 job에 대해 종료 코드
    1을 돌려줍니다. 그 줄을 그대로 찍으면 셀 출력 맨 앞에 "[rc=1]"이 남아서, 실제 실패로 읽히는
    오해를 부릅니다.
    """
    if isinstance(cmd, str):
        cmd = shlex.split(cmd)
    cmd = [str(x) for x in cmd]
    if echo:
        print("$ " + " ".join(shlex.quote(x) for x in cmd), flush=True)
    t0 = time.time()
    proc = subprocess.Popen(
        cmd, cwd=str(PROJECT_ROOT), stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT, text=True, encoding="utf-8",
        errors="replace", bufsize=1,
    )
    buf = []
    for line in proc.stdout:
        buf.append(line)
        if not quiet:
            print(line, end="", flush=True)
    rc = proc.wait()
    dt = time.time() - t0
    if not quiet:
        print("[rc=%d] %.1f초" % (rc, dt), flush=True)
    if rc != 0:
        if hint:
            print("실패했습니다. 같은 명령에 --help 를 붙여서 인자 이름부터 확인하십시오.", flush=True)
        if check:
            raise RuntimeError("명령이 실패했습니다: " + " ".join(cmd))
    return {"rc": rc, "seconds": dt, "output": "".join(buf)}


def vqa(module, *args, **kw):
    """python -m vqa.<module> 을 실행합니다."""
    return run([PY, "-m", "vqa." + module] + [str(a) for a in args], **kw)


def script(name, *args, **kw):
    """scripts/<name> 을 실행합니다."""
    return run([PY, str(Path("scripts") / name)] + [str(a) for a in args], **kw)


def read_csv_rows(path):
    with open(path, newline="", encoding="utf-8") as fh:
        return list(csv.DictReader(fh))


def read_ids(path):
    return [ln.strip() for ln in Path(path).read_text(encoding="utf-8").splitlines() if ln.strip()]


def sha256_of(path, limit=None):
    h = hashlib.sha256()
    with open(path, "rb") as fh:
        for chunk in iter(lambda: fh.read(1 << 20), b""):
            h.update(chunk)
    digest = h.hexdigest()
    return digest[:limit] if limit else digest


print("PROJECT_ROOT      :", PROJECT_ROOT)
for _name in ("VQA_DATA_ROOT", "VQA_RUN_ROOT", "VQA_LOCAL_SCRATCH", "VQA_MODEL_ROOT"):
    print("%-18s: %s" % (_name, os.environ.get(_name, "(미설정)")))

# --- 어떤 코드로 실행했는지 남깁니다 ---
_code_version_path = RUN_ROOT_P / "code_version.txt"
with _code_version_path.open("a", encoding="utf-8") as _fh:
    _fh.write("%s\t%s\n" % (time.strftime("%Y-%m-%d %H:%M:%S"),
                            CODE_VERSION or "(알 수 없음)"))
print("코드 버전을 기록했습니다: %s -> %s" % (_code_version_path, CODE_VERSION))

PROJECT_ROOT      : /content/ssafy-16-2-ai
VQA_DATA_ROOT     : /content/data
VQA_RUN_ROOT      : /content/drive/MyDrive/vqa/runs
VQA_LOCAL_SCRATCH : /content/scratch
VQA_MODEL_ROOT    : /content/drive/MyDrive/vqa/hf_cache
코드 버전을 기록했습니다: /content/drive/MyDrive/vqa/runs/code_version.txt -> adc3727407a0984c44ea0409b8c44ba72beee075


## 4. 환경 설치와 확인

Colab은 노트북 하나마다 가상 머신을 하나씩 따로 배정합니다. 그래서 20번 노트북에서 설치를
끝냈더라도 이 노트북의 가상 머신에는 아무 패키지도 깔려 있지 않습니다. 2026-09-21에 27B
작업이 `Using bitsandbytes 4-bit quantization requires bitsandbytes` 오류로 세 번 죽은 원인이
바로 이것입니다.

작업을 시작하기 전에 런타임 메뉴의 세션 관리 화면에서 다른 노트북의 세션을 반드시 종료하십시오.
A100 두 대가 동시에 돌면 compute unit이 두 배로 빠지고, 80GB를 배정받을 확률도 떨어집니다.

아래 셀은 가속 커널인 fla도 함께 설치하고 확인합니다. Qwen3.5의 Gated DeltaNet은 이 패키지가
있으면 전용 커널을 쓰고, 없으면 PyTorch 참조 구현으로 내려갑니다. 2026-09-21 Colab A100에서
bf16 학습이 microbatch당 7.3초로 참고 실행의 3.37초보다 두 배 이상 느렸던 원인이 이
fallback이었습니다. 설치에 실패하더라도 학습 자체는 가능하므로 셀은 멈추지 않고 경고만 크게
출력합니다. 그 경고를 보면 일정을 두 배로 잡아야 할지 먼저 판단하십시오.

In [5]:
# --- 환경 설치와 확인 (20번 노트북과 21번 노트북이 같은 셀을 씁니다) ---
# Colab은 노트북마다 가상 머신을 따로 배정합니다. 20번에서 설치를 끝냈더라도 21번 가상 머신에는
# 아무것도 깔려 있지 않으므로, 새 가상 머신을 받을 때마다 이 셀을 처음부터 다시 실행해야 합니다.
REQ = "requirements/colab.txt"
REQ_NODEPS = "requirements/colab-nodeps.txt"
for _req in (REQ, REQ_NODEPS):
    if not Path(_req).exists():
        raise RuntimeError(_req + " 이 없습니다. 코드 가져오기 셀이 제대로 끝났는지 먼저 확인하십시오.")

_TORCH_VERSION_SRC = "import torch; print(torch.__version__)"


def _torch_version():
    """지금 이 런타임에 실제로 설치되어 있는 torch 버전을 별도 프로세스에서 읽습니다."""
    res = run([PY, "-c", _TORCH_VERSION_SRC], check=False, quiet=True, echo=False, hint=False)
    return res["output"].strip().splitlines()[-1].strip() if res["rc"] == 0 else None


TORCH_BEFORE = _torch_version()
print("설치 전 torch :", TORCH_BEFORE or "(확인하지 못했습니다)")

# Colab 기본 이미지에는 torchao 0.10.0이 들어 있습니다. PEFT가 비양자화 Linear에 LoRA를 붙일 때
# 이 버전과 충돌해서 실패하므로 bf16 경로에서는 반드시 먼저 지웁니다. 설치되어 있지 않으면
# 아무 일도 일어나지 않습니다.
run([PY, "-m", "pip", "uninstall", "-y", "torchao"], check=False, quiet=True)

_install = run([PY, "-m", "pip", "install", "-q", "-r", REQ], check=False)
if _install["rc"] != 0:
    raise RuntimeError(
        "requirements/colab.txt 설치가 실패했습니다. 위 출력에서 어느 패키지가 막혔는지 확인한 다음 "
        "이 셀을 다시 실행하십시오."
    )

# fla 계열은 자기 의존성으로 torch와 triton을 다시 끌어옵니다. 그대로 두면 이미 통과해 둔
# torch 빌드가 바뀌므로 반드시 --no-deps로 설치합니다. 이 단계가 실패하더라도 학습 자체는
# 가능하므로 셀을 멈추지 않고, 아래에서 경고만 크게 출력합니다.
FLA_OK = True
FLA_REASON = ""
_install_nodeps = run([PY, "-m", "pip", "install", "-q", "--no-deps", "-r", REQ_NODEPS], check=False)
if _install_nodeps["rc"] != 0:
    FLA_OK = False
    FLA_REASON = "requirements/colab-nodeps.txt 설치가 실패했습니다."

TORCH_AFTER = _torch_version()
print("설치 후 torch :", TORCH_AFTER or "(확인하지 못했습니다)")
if TORCH_BEFORE and TORCH_AFTER and TORCH_BEFORE != TORCH_AFTER:
    raise RuntimeError(
        "설치 과정에서 torch 버전이 %s 에서 %s 로 바뀌었습니다. --no-deps 설치가 의도대로 "
        "동작하지 않은 것입니다. requirements/colab-nodeps.txt 에 --no-deps 없이 설치하면 안 되는 "
        "패키지만 들어 있는지 확인하고, 런타임을 재시작한 다음 이 셀부터 다시 실행하십시오."
        % (TORCH_BEFORE, TORCH_AFTER)
    )

# 점검은 반드시 별도 프로세스에서 수행합니다. 지금 커널은 예전 버전을 이미 import 해 두었을 수
# 있고, 그러면 화면에는 옛 버전이 그대로 찍혀서 문제를 놓치게 됩니다.
_ENV_CHECK_SRC = """
import importlib
import torch

print("torch                :", torch.__version__)
print("torch.cuda.available :", torch.cuda.is_available())
if torch.cuda.is_available():
    _props = torch.cuda.get_device_properties(0)
    print("gpu                  :", _props.name)
    print("vram_gb              : %.1f" % (_props.total_memory / 1024 ** 3))
for _name in ("bitsandbytes", "transformers", "peft", "accelerate"):
    _mod = importlib.import_module(_name)
    print("%-21s: %s" % (_name, getattr(_mod, "__version__", "(버전 정보 없음)")))
try:
    import torchao
except ImportError:
    print("torchao              : 설치되어 있지 않습니다 (정상)")
else:
    raise AssertionError(
        "torchao가 아직 설치되어 있습니다(%s). PEFT가 bf16 LoRA를 붙일 때 실패합니다. "
        "pip uninstall -y torchao 를 다시 실행한 다음 런타임을 재시작하십시오."
        % getattr(torchao, "__version__", "버전 미상")
    )
if not torch.cuda.is_available():
    raise AssertionError("CUDA를 쓸 수 없습니다. 런타임 유형이 GPU인지 확인하십시오.")
print("환경 확인을 통과했습니다.")
"""

_verify = run([PY, "-c", _ENV_CHECK_SRC], check=False, echo=False)
if _verify["rc"] != 0:
    raise RuntimeError(
        "환경 확인이 실패했습니다. 바로 위 메시지를 그대로 읽고 원인을 해결한 다음 이 셀을 다시 "
        "실행하십시오. torchao가 남아 있다는 메시지가 보이면 런타임을 재시작하고 이 셀부터 다시 "
        "실행하면 됩니다."
    )

# --- 가속 커널(fla) 확인 ---
# transformers의 Qwen3.5 Gated DeltaNet은 fla가 있으면 그 커널을 쓰고, 없으면 PyTorch 참조
# 구현으로 내려갑니다. 2026-09-21 Colab A100에서 bf16 학습이 microbatch당 7.3초로 참고 실행의
# 3.37초보다 두 배 이상 느렸던 원인이 이 fallback이었습니다. 여기에서는 import가 되는지와,
# 작은 텐서 하나로 forward와 backward가 유한한 값을 내는지까지 확인합니다.
_FLA_CHECK_SRC = """
import torch
import torch.nn.functional as F
from fla.ops.gated_delta_rule import chunk_gated_delta_rule

try:
    import fla
    print('fla                  :', getattr(fla, '__version__', '(버전 정보 없음)'))
except Exception:
    pass
print('chunk_gated_delta_rule import 성공')

if not torch.cuda.is_available():
    raise AssertionError('CUDA가 없어 fla 커널을 시험할 수 없습니다.')

B, T, H, D = 1, 1024, 8, 128
dev = 'cuda'
torch.manual_seed(0)
q = torch.randn(B, T, H, D, device=dev, dtype=torch.bfloat16, requires_grad=True)
k = torch.randn(B, T, H, D, device=dev, dtype=torch.bfloat16, requires_grad=True)
v = torch.randn(B, T, H, D, device=dev, dtype=torch.bfloat16, requires_grad=True)
# g는 float32 음수 decay, beta는 0과 1 사이의 값입니다. 모델이 넘기는 형태와 같게 맞춥니다.
g = -F.softplus(torch.randn(B, T, H, device=dev, dtype=torch.float32))
beta = torch.rand(B, T, H, device=dev, dtype=torch.bfloat16)

# delta rule은 k의 노름이 1보다 크면 상태가 발산해 T=1024에서 inf/nan이 나옵니다. 실제 모델도
# q, k를 L2 정규화해서 넘기므로 여기서도 정규화한 값을 넘깁니다. 정규화하지 않으면 fla가
# 정상인데도 이 확인이 실패해 거짓 경고가 뜹니다. q, k는 leaf로 두어 grad 확인은 그대로 됩니다.
qn = F.normalize(q, p=2, dim=-1)
kn = F.normalize(k, p=2, dim=-1)

try:
    out = chunk_gated_delta_rule(qn, kn, v, g=g, beta=beta)
except TypeError:
    # 버전에 따라 g와 beta를 위치 인자로만 받습니다.
    out = chunk_gated_delta_rule(qn, kn, v, g, beta)
o = out[0] if isinstance(out, (tuple, list)) else out
assert torch.isfinite(o.float()).all(), 'forward 결과에 유한하지 않은 값이 있습니다.'
o.float().pow(2).mean().backward()
for name, tensor in (('q', q), ('k', k), ('v', v)):
    assert tensor.grad is not None, name + ' 의 grad가 없습니다.'
    assert torch.isfinite(tensor.grad.float()).all(), name + ' 의 grad에 유한하지 않은 값이 있습니다.'
print('fla 커널 forward/backward 정상 (B=%d T=%d H=%d D=%d, bf16)' % (B, T, H, D))
"""

if FLA_OK:
    _fla = run([PY, "-c", _FLA_CHECK_SRC], check=False, echo=False)
    if _fla["rc"] != 0:
        FLA_OK = False
        FLA_REASON = "fla 커널 확인이 실패했습니다. 바로 위 출력에 원인이 있습니다."

if FLA_OK:
    print()
    print("가속 커널(fla)을 쓸 수 있습니다. Gated DeltaNet이 참조 구현으로 내려가지 않습니다.")
else:
    print()
    print("!" * 78)
    print("경고: 가속 커널(fla)을 쓸 수 없습니다.")
    print("사유: %s" % (FLA_REASON or "원인을 특정하지 못했습니다."))
    print("")
    print("이 상태로도 학습과 추론은 동작합니다. 다만 Qwen3.5의 Gated DeltaNet이 PyTorch 참조")
    print("구현으로 내려가기 때문에 학습이 약 2배 느려집니다. 10시간이 걸릴 학습이라면 20시간이")
    print("되므로, 일정을 그대로 두고 진행할 수 있는지 먼저 판단하십시오.")
    print("")
    print("다시 시도하려면 아래 두 줄을 차례로 실행한 다음 이 셀을 다시 실행하십시오.")
    print("  !pip install -q --no-deps fla-core flash-linear-attention")
    print("  !pip install -q einops")
    print("")
    print("preflight의 kernel_fast_path 검사가 같은 내용을 WARN으로 다시 알려 줍니다.")
    print("!" * 78)

설치 전 torch : 2.11.0+cu128
$ /usr/bin/python3 -m pip uninstall -y torchao
$ /usr/bin/python3 -m pip install -q -r requirements/colab.txt
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 63.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 296.7/296.7 kB 30.7 MB/s eta 0:00:00
[rc=0] 4.2초
$ /usr/bin/python3 -m pip install -q --no-deps -r requirements/colab-nodeps.txt
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.3/45.3 kB 3.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.6/45.6 kB 4.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 819.2/819.2 kB 40.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 399.6/399.6 kB 43.6 MB/s eta 0:00:00
[rc=0] 2.2초
설치 후 torch : 2.11.0+cu128
torch                : 2.11.0+cu128
torch.cuda.available : True
gpu                  : NVIDIA A100-SXM4-80GB
vram_gb              : 79.3
bitsandbytes         : 0.50.2
transformers         : 5.16.1
peft                 : 0.20.0
accelerate           : 1.

## 5. GPU 확인과 정밀도, run-id 결정

정밀도는 20번 노트북이 남긴 `colab_choice.json` 이 아니라 지금 이 가상 머신의 VRAM을 직접
재서 정합니다. 20번이 40GB에서 고른 nf4가 80GB 가상 머신까지 따라와서 조용히 nf4로 돌아간
사고가 있었기 때문입니다. `colab_choice.json` 은 참고 정보로만 출력합니다.

### 설정 파일과 정밀도를 따로 정하는 까닭

파라미터 셀에서 `CONFIG` 를 직접 적었다면 그 파일이 언제나 우선합니다. 다만 VRAM이 모자라서
정밀도를 낮춰야 하는 상황에서 설정 파일 자체를 nf4 설정으로 바꿔 버리면, 그 파일에 적어 둔
해상도까지 함께 바뀌어 버립니다. 확대본을 고른 실행에서 바로 이 일이 일어나면 `min_pixels`
상향이 조용히 사라지고, 실제로는 기본 해상도로 돌면서 결과만 확대본이라고 기록됩니다. 그래서
이 셀은 설정 파일을 바꾸지 않고 `--set precision=nf4` 를 입력 설정에 덧붙이는 방식으로 정밀도만
내립니다. `CONFIG` 를 비워 두었을 때에는 예전처럼 nf4 설정 파일을 그대로 고릅니다.

### VRAM에 따른 동작

| 감지된 VRAM | 학습 | 추론 |
|---|---|---|
| 70GB 이상 | bf16으로 허용합니다. | bf16으로 수행합니다. |
| 35GB 이상 70GB 미만 | 기본 해상도 nf4 설정에서만 허용합니다. | nf4로 수행합니다. |
| 20GB 이상 35GB 미만 | 막습니다. | nf4로 수행합니다. |
| 20GB 미만 | 오류를 내고 멈춥니다. | 수행하지 않습니다. |

24GB짜리 L4에서도 27B 추론은 들어갑니다. nf4로 양자화한 가중치가 약 15GB이고, 여기에 시각
토큰 2,048개 분량의 활성값과 KV 캐시를 더해도 20GB를 넘지 않기 때문입니다. bf16 참고 실행이
가중치 54GB에 peak 57GB였다는 기록도 활성값이 3GB 남짓이었다는 뜻이므로 같은 결론을 가리킵니다.
다만 학습은 옵티마이저 상태와 역전파 활성값이 더 붙으므로 24GB에서는 허용하지 않습니다.

### 모델 식별자는 설정 파일에서 읽습니다

이 셀은 `MODEL_ID` 와 `MODEL_REVISION` 을 파라미터 셀의 값이 아니라 확정된 `CONFIG` 에서
다시 읽습니다. 같은 값을 두 곳에 적어 두면 한쪽만 고쳤을 때 5-1절이 내려받는 가중치와 추론
job이 실제로 올리는 가중치가 어긋나기 때문입니다. 설정 파일에 `revision` 이 없으면 `main` 을
쓰되 경고를 출력합니다.

run-id는 모델과 프롬프트와 해상도와 정밀도가 다른 결과가 한 폴더에 섞이지 않도록 다음
규칙으로 만듭니다.

- 기본 이름의 앞머리 `colab27b` 는 Qwen3.5-27B를 가리킵니다. `CONFIG` 의 `model_id` 가
  Qwen3.8-27B이면 이 부분을 `colab38_27b` 로 바꿉니다. 프롬프트와 해상도가 같으면 뒤에 붙는
  접미사가 모두 같아지므로, 앞머리를 구분하지 않으면 두 모델의 예측이 한 폴더에 섞입니다.
- `TRAIN_SCOPE` 가 `full` 이면 학습 run-id의 기본 이름 뒤에 먼저 `_full` 을 붙입니다.
- `PROMPT_ID` 가 `v1_image_only` 이면 프롬프트 접미사를 붙이지 않고 기본 이름을 그대로 씁니다.
- 그 밖의 프롬프트이면 `_<PROMPT_ID>` 를 붙입니다.
- `CONFIG` 에 확대본(`configs/colab_qwen35_27b_bf16_upscale.yaml` 또는
  `configs/colab_qwen38_27b_bf16_upscale.yaml`)을 적었으면 `_upscale` 을 덧붙입니다. 해상도가
  다른 예측이 같은 run 폴더에 섞이지 않게 하려는 것입니다.
- 정책이 bf16인데 VRAM이 모자라서 nf4로 내려가면 맨 뒤에 `_nf4` 를 덧붙이고 학습을 막습니다.

예를 들어 `TRAIN_SCOPE` 가 `full` 이고 프롬프트가 기본값이며 bf16으로 돌아가면 학습 run-id는
`colab27b_ft_bf16_full` 이 됩니다. 같은 조건에서 확대본을 쓰고 프롬프트를 `v2_read_text`
로 바꾸면 `colab27b_ft_bf16_full_v2_read_text_upscale` 이 됩니다. 40GB나 24GB를 받아서 nf4로
내려간 원본 추론은 기본 설정에서 `colab27b_base_bf16_nf4` 가 되고, 확대본에서는
`colab27b_base_bf16_upscale_nf4` 가 됩니다. 원본 모델의 run-id는 학습과 무관하므로 `_full` 이
붙지 않습니다.

Qwen3.8 비교 실험도 같은 규칙을 따르되 앞머리만 달라집니다. `CONFIG = QWEN38_UPSCALE_CONFIG`
에 프롬프트를 `v2_read_text` 로 두고 `TRAIN_SCOPE` 가 `fold0` 이면 원본 추론은
`colab38_27b_base_bf16_v2_read_text_upscale`, 학습은
`colab38_27b_ft_bf16_v2_read_text_upscale` 이 됩니다. `TRAIN_SCOPE` 를 `full` 로 바꾸면 학습
run-id만 `colab38_27b_ft_bf16_full_v2_read_text_upscale` 이 되고 원본 추론의 이름은 그대로입니다.

확대본은 평균 시각 토큰이 1.46배가 되므로 VRAM이 70GB에 못 미치는 가상 머신에서는 정밀도를
낮춰도 학습에는 위험합니다. 그래서 이 셀은 VRAM이 70GB 미만인데 확대본이나 bf16 설정이
선택되면 학습을 막고 경고를 출력합니다. 추론은 그대로 진행할 수 있습니다. 이 셀은
`TRAIN_SCOPE` 에 맞는 분할 파일이 실제로 있는지도 함께 확인합니다. 파일이 없으면
`python scripts/folds_to_split.py` 를 먼저 실행하십시오.

In [6]:
import shutil

# --- 가중치 캐시 위치를 정합니다 ---
def _free_gb(path):
    """해당 경로가 속한 파일 시스템의 여유 공간을 GB 단위로 돌려줍니다."""
    try:
        target = Path(path)
        target.mkdir(parents=True, exist_ok=True)
        return round(shutil.disk_usage(str(target)).free / 1024 ** 3, 1)
    except Exception as exc:
        print("%s 의 여유 공간을 확인하지 못했습니다: %s" % (path, exc))
        return 0.0


if MODEL_CACHE == "drive":
    CACHE_ROOT = MODEL_ROOT
    print("MODEL_CACHE 가 \"drive\" 이므로 Drive 캐시를 사용합니다:", CACHE_ROOT)
elif MODEL_CACHE == "local":
    CACHE_ROOT = LOCAL_MODEL_ROOT
    print("MODEL_CACHE 가 \"local\" 이므로 가상 머신 로컬 디스크를 사용합니다:", CACHE_ROOT)
elif MODEL_CACHE == "auto":
    _snap_path = RUN_ROOT_P / "model_snapshot.json"
    _snap_cache = ""
    if _snap_path.exists():
        _snap_cache = json.loads(_snap_path.read_text(encoding="utf-8")).get("cache_dir") or ""
    if _snap_cache and Path(_snap_cache).is_dir():
        CACHE_ROOT = _snap_cache
        print("20번 노트북이 쓴 가중치 캐시를 그대로 사용합니다:", CACHE_ROOT)
    else:
        _drive_free = _free_gb(MODEL_ROOT)
        if _drive_free >= MODEL_CACHE_NEED_GB:
            CACHE_ROOT = MODEL_ROOT
            print("Drive 여유가 %.1fGB이므로 Drive 캐시를 사용합니다: %s" % (_drive_free, CACHE_ROOT))
        else:
            CACHE_ROOT = LOCAL_MODEL_ROOT
            print("Drive 여유가 %.1fGB뿐이라 %.0fGB에 못 미치므로 로컬 디스크 캐시를 사용합니다: %s"
                  % (_drive_free, MODEL_CACHE_NEED_GB, CACHE_ROOT))
            print("런타임이 회수되면 가중치가 사라지므로, 다시 접속할 때마다 내려받아야 합니다.")
else:
    raise RuntimeError("MODEL_CACHE 는 auto, drive, local 중 하나여야 합니다: %r" % (MODEL_CACHE,))

MODEL_ROOT = CACHE_ROOT
MODEL_ROOT_P = Path(MODEL_ROOT)
MODEL_ROOT_P.mkdir(parents=True, exist_ok=True)
os.environ["VQA_MODEL_ROOT"] = str(MODEL_ROOT_P)
os.environ["HF_HOME"] = str(MODEL_ROOT_P)
# HF_HOME만 지정하면 스냅샷이 HF_HOME/hub 아래에 놓입니다. 20번 노트북의 다운로드 셀은
# --cache-dir 로 같은 경로를 지정하므로, 두 값을 맞춰 두어야 같은 가중치를 두 번 받지 않습니다.
os.environ["HF_HUB_CACHE"] = str(MODEL_ROOT_P)

CACHE_FREE_GB = _free_gb(MODEL_ROOT)
print("캐시 여유 공간 : %.1fGB (27B 원본 스냅샷은 약 54GB입니다)" % CACHE_FREE_GB)

# --- 지금 이 가상 머신의 VRAM을 직접 잽니다 ---
VRAM_GB = 0.0
GPU_NAME = None
_smi = run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"],
           check=False, quiet=True, echo=False, hint=False)
if _smi["rc"] == 0 and _smi["output"].strip():
    _parts = [x.strip() for x in _smi["output"].strip().splitlines()[0].split(",")]
    GPU_NAME = _parts[0]
    VRAM_GB = round(int(_parts[1].split()[0]) / 1024.0, 1)
else:
    print("nvidia-smi를 쓸 수 없습니다. 런타임 유형이 GPU인지 확인하십시오.")

# --- VRAM 하한을 먼저 확인합니다 ---
# nf4로 양자화한 27B 가중치가 약 15GB이므로, 활성값과 KV 캐시까지 얹으려면 20GB는 있어야
# 합니다. 이보다 작으면 추론조차 완주하지 못하므로 여기에서 멈춥니다.
VRAM_MIN_INFER_GB = 20.0
VRAM_MIN_TRAIN_GB = 35.0
VRAM_BF16_GB = 70.0

if VRAM_GB < VRAM_MIN_INFER_GB:
    raise RuntimeError(
        "감지된 VRAM이 %.1fGB입니다. 27B는 nf4로 양자화해도 가중치만 약 15GB이므로, "
        "%.0fGB 미만에서는 추론도 완주하지 못합니다. 런타임 유형이 GPU인지 먼저 확인하시고, "
        "GPU가 맞다면 런타임을 끊고 다시 연결해서 더 큰 GPU를 받으십시오. 계속 작은 GPU만 "
        "배정된다면 docs/README_team.md 의 9B 경로로 전환해야 합니다."
        % (VRAM_GB, VRAM_MIN_INFER_GB)
    )

# --- 정밀도를 정합니다 ---
# 설정 파일을 바꾸는 대신 정밀도만 내릴 수 있도록, 먼저 "nf4가 필요한가"만 판정합니다.
PRECISION_SUFFIX = ""
TRAINING_ALLOWED = True

if PRECISION_POLICY == "nf4":
    WANT_NF4 = True
elif PRECISION_POLICY in ("auto", "bf16"):
    WANT_NF4 = VRAM_GB < VRAM_BF16_GB
else:
    raise RuntimeError(
        "PRECISION_POLICY 는 bf16, auto, nf4 중 하나여야 합니다: %r" % (PRECISION_POLICY,))

if WANT_NF4 and PRECISION_POLICY == "bf16":
    PRECISION_SUFFIX = "_nf4"
    TRAINING_ALLOWED = False
    print("=" * 72)
    print("경고: 팀 정책은 bf16인데 이 GPU의 VRAM은 %.1fGB뿐입니다." % VRAM_GB)
    print("이 가상 머신에서는 bf16 학습을 시작할 수도, 이어서 할 수도 없습니다.")
    print("80GB A100을 받으려면 런타임을 끊고 다시 연결하십시오.")
    print("계속 진행하면 추론만 nf4로 돌리며, run-id 뒤에 \"_nf4\"를 붙여서")
    print("bf16 결과와 캐시가 절대 섞이지 않게 합니다.")
    print("=" * 72)

# 파라미터 셀에서 CONFIG를 직접 적었다면 그 파일이 언제나 우선합니다. 비워 두었을 때에만
# 정밀도에 맞는 설정 파일을 고릅니다.
if CONFIG:
    RESOLVED_CONFIG = CONFIG
    print("파라미터 셀의 CONFIG를 그대로 사용합니다:", RESOLVED_CONFIG)
else:
    RESOLVED_CONFIG = NF4_CONFIG if WANT_NF4 else BF16_CONFIG
CONFIG = RESOLVED_CONFIG
if not Path(CONFIG).exists():
    raise RuntimeError("config 파일이 없습니다: %s" % CONFIG)

# --- 확정된 설정 파일에서 정밀도와 해상도를 다시 읽습니다 ---
# CONFIG를 직접 적었을 때에도 화면에 출력하는 정밀도와 run-id가 실제 설정과 어긋나지 않게
# 하려는 것입니다. 파일 이름이 아니라 설정 로더를 거치므로 값이 유효한지도 여기에서 걸러집니다.
from vqa import config as _config_mod


def precision_overrides_for(config_path):
    """설정 파일의 정밀도가 이 GPU에 맞지 않으면 덮어쓸 인자를 돌려줍니다.

    설정 파일을 nf4 파일로 바꿔 버리면 그 파일에 적힌 해상도까지 함께 바뀝니다. 확대본을
    고른 실행에서 그렇게 하면 min_pixels 상향이 조용히 사라지므로, 파일은 그대로 두고
    정밀도만 --set 으로 내립니다.
    """
    file_precision = str(_config_mod.get_path(_config_mod.load_config(config_path), "precision"))
    if WANT_NF4 and file_precision != "nf4":
        return ["--set", "precision=nf4"]
    return []


PRECISION_OVERRIDES = precision_overrides_for(CONFIG)
_cfg_preview = _config_mod.load_config(CONFIG, PRECISION_OVERRIDES[1:] or None)
PRECISION = str(_config_mod.get_path(_cfg_preview, "precision"))
MIN_PIXELS = int(_config_mod.get_path(_cfg_preview, "min_pixels"))
MAX_PIXELS = int(_config_mod.get_path(_cfg_preview, "max_pixels"))
if PRECISION_OVERRIDES:
    print("설정 파일은 그대로 두고 정밀도만 내립니다: %s" % " ".join(PRECISION_OVERRIDES))

# --- 모델 식별자와 revision은 언제나 확정된 설정 파일에서 읽습니다 ---
# 파라미터 셀에 적힌 값과 CONFIG 가 어긋나면 5-1절이 엉뚱한 가중치를 내려받고, 그 뒤의 job은
# 설정 파일의 모델을 다시 받게 됩니다. 같은 값을 두 곳에 적어 두지 않으려고 여기에서 덮어씁니다.
_cfg_model_id = str(_config_mod.get_path(_cfg_preview, "model_id"))
_cfg_revision = _config_mod.get_path(_cfg_preview, "revision")
if _cfg_model_id != MODEL_ID:
    print("MODEL_ID 를 설정 파일의 값으로 바꿉니다: %s -> %s" % (MODEL_ID, _cfg_model_id))
MODEL_ID = _cfg_model_id
if _cfg_revision:
    MODEL_REVISION = str(_cfg_revision)
else:
    MODEL_REVISION = "main"
    print("경고: %s 에 revision 이 없어서 main 을 씁니다. 나중에 같은 가중치를 다시 받을 수 "
          "없으므로, 커밋 해시를 설정 파일에 적어 두십시오." % CONFIG)

# 확대본을 쓰면 run-id에 "_upscale"을 붙여서 기존 결과와 섞이지 않게 합니다.
UPSCALE = "upscale" in Path(CONFIG).stem
RES_SUFFIX = "_upscale" if UPSCALE else ""

# 27B 학습은 nf4로 낮춰도 40GB급 VRAM이 필요합니다. 그보다 작은 GPU에서는 추론만 허용합니다.
if VRAM_GB < VRAM_MIN_TRAIN_GB:
    TRAINING_ALLOWED = False
    print("=" * 72)
    print("안내: VRAM이 %.1fGB이므로 이 가상 머신에서는 학습을 허용하지 않습니다." % VRAM_GB)
    print("27B 학습은 nf4로 낮춰도 %.0fGB 이상이 필요합니다. 추론은 그대로 진행할 수 있습니다."
          % VRAM_MIN_TRAIN_GB)
    print("=" * 72)

# 확대본은 평균 시각 토큰이 1.46배이므로 40GB 가상 머신에서는 nf4로도 학습이 위험합니다.
# 정밀도 정책이 무엇이든, 여기에서 한 번 더 학습을 막습니다.
if VRAM_GB < VRAM_BF16_GB and (UPSCALE or PRECISION == "bf16"):
    TRAINING_ALLOWED = False
    print("=" * 72)
    print("경고: 설정 %s 는 VRAM이 70GB 이상일 때를 전제로 합니다." % CONFIG)
    print("이 가상 머신의 VRAM은 %.1fGB뿐이므로 학습을 시작할 수도, 이어서 할 수도 없습니다." % VRAM_GB)
    if UPSCALE:
        print("확대본은 평균 시각 토큰이 1.46배여서 정밀도를 낮춰도 학습 활성값이 들어가지 않습니다.")
    print("추론은 그대로 진행할 수 있습니다. 학습까지 하려면 80GB A100을 받아야 합니다.")
    print("=" * 72)

# --- 학습 범위를 정합니다 ---
# "full" 은 fold 0까지 학습에 넣으므로, 그 run의 fold 0 정확도는 학습 데이터를 다시 맞힌
# 점수가 됩니다. 15번 절과 16번 절은 그 사실을 알리고 건너뜁니다.
if TRAIN_SCOPE == "fold0":
    SPLIT_FILE, SCOPE_SUFFIX = SPLIT_FILE_FOLD0, ""
elif TRAIN_SCOPE == "full":
    SPLIT_FILE, SCOPE_SUFFIX = SPLIT_FILE_FULL, "_full"
else:
    raise RuntimeError("TRAIN_SCOPE 는 fold0 또는 full 이어야 합니다: %r" % (TRAIN_SCOPE,))
if not Path(SPLIT_FILE).exists():
    raise RuntimeError(
        "분할 파일이 없습니다: %s\n"
        "`python scripts/folds_to_split.py` 를 먼저 실행하십시오." % SPLIT_FILE)

# --- run-id를 확정합니다 ---
# 모델이 다르면 run 폴더 이름의 앞머리부터 달라야 합니다. 같은 프롬프트와 같은 해상도로
# Qwen3.5와 Qwen3.8을 번갈아 돌리면 접미사가 모두 같아져서, 앞머리를 구분하지 않으면 두
# 모델의 예측이 한 폴더에 섞이기 때문입니다. 기본 이름이 아직 Qwen3.5의 이름 그대로일
# 때에만 바꾸므로, 파라미터 셀에서 이름을 직접 고쳤다면 그 값을 그대로 존중합니다.
MODEL_RUN_PREFIX = _config_mod.run_id_prefix(MODEL_ID)
if MODEL_RUN_PREFIX != _config_mod.DEFAULT_RUN_ID_PREFIX:
    for _name in ("BASE_RUN_ID", "FT_RUN_ID"):
        _stem = globals()[_name]
        if _stem.startswith(MODEL_RUN_PREFIX):
            # 이 셀만 다시 실행한 경우입니다. 앞머리가 이미 바뀌어 있으므로 아무 일도 하지 않습니다.
            pass
        elif _stem.startswith(_config_mod.DEFAULT_RUN_ID_PREFIX):
            _new = MODEL_RUN_PREFIX + _stem[len(_config_mod.DEFAULT_RUN_ID_PREFIX):]
            globals()[_name] = _new
            print("%s 를 모델에 맞게 바꿉니다: %s -> %s" % (_name, _stem, _new))
        else:
            print("%s 가 %r 로 시작하지 않아 이름을 그대로 둡니다: %s"
                  % (_name, _config_mod.DEFAULT_RUN_ID_PREFIX, _stem))

PROMPT_SUFFIX = "" if PROMPT_ID == "v1_image_only" else "_" + PROMPT_ID


def resolve_run_id(stem, prompt_id=None, res_suffix=None):
    """기본 이름에 프롬프트, 해상도, 정밀도 접미사를 차례로 붙여 실제 run-id를 만듭니다.

    해상도를 정밀도보다 앞에 붙이는 까닭은, 기본 해상도에서 nf4로 내려간 기존 run-id를
    그대로 두면서 확대본에서 nf4로 내려간 run-id를 "_upscale_nf4" 로 읽히게 하기
    위해서입니다.
    """
    suffix = "" if (prompt_id or PROMPT_ID) == "v1_image_only" else "_" + (prompt_id or PROMPT_ID)
    res = RES_SUFFIX if res_suffix is None else res_suffix
    return stem + suffix + res + PRECISION_SUFFIX


BASE_RUN = resolve_run_id(BASE_RUN_ID)
# 학습 범위 접미사는 기본 이름에 먼저 붙입니다. 그래야 전체 데이터로 학습한 결과가 fold 0으로
# 학습한 기존 run 폴더에 섞이지 않습니다.
FT_RUN = resolve_run_id(FT_RUN_ID + SCOPE_SUFFIX)
CMP_RUN = resolve_run_id(BASE_RUN_ID, COMPARE_PROMPT_ID)

# 해상도 비교(10-1절)에 쓰는 run-id입니다. 프롬프트와 정밀도는 9번 절과 같게 두고 해상도만
# 다르게 두어야, 두 결과의 차이를 해상도 하나로 설명할 수 있습니다.
RES_CMP_UPSCALE = "upscale" in Path(RESOLUTION_COMPARE_CONFIG).stem
RES_CMP_RUN = resolve_run_id(BASE_RUN_ID, res_suffix="_upscale" if RES_CMP_UPSCALE else "")

# 추론에 쓸 adapter를 정합니다. ADAPTER_RUN_ID 를 적으면 다른 run이 학습한 adapter를 빌려
# 오고, 결과 파일에는 "_xadapter" 접미사를 붙여서 이 run이 직접 학습한 결과와 구분합니다.
if ADAPTER_RUN_ID:
    ADAPTER_SOURCE_RUN = ADAPTER_RUN_ID
    XADAPTER_SUFFIX = "_xadapter"
else:
    ADAPTER_SOURCE_RUN = FT_RUN
    XADAPTER_SUFFIX = ""
ADAPTER_PATH = str(RUN_ROOT_P / ADAPTER_SOURCE_RUN / "final_adapter")
if ADAPTER_RUN_ID and not Path(ADAPTER_PATH).is_dir():
    raise RuntimeError(
        "ADAPTER_RUN_ID 로 지정한 run의 final_adapter 를 찾지 못했습니다: %s\n"
        "run-id의 철자와 Drive 연결 상태를 확인하시고, 그 run의 학습이 끝났는지 먼저 "
        "확인하십시오. 이 값을 비워 두면 이 노트북이 학습한 %s 의 adapter를 씁니다."
        % (ADAPTER_PATH, FT_RUN)
    )
# 결과 파일 이름에도 접미사를 붙여서 같은 run 폴더 안에서 섞이지 않게 합니다. 이 셀만 다시
# 실행해도 접미사가 두 번 붙지 않도록 이미 붙어 있는지 확인합니다.
if XADAPTER_SUFFIX and not VAL_OUT_FT.endswith(XADAPTER_SUFFIX):
    VAL_OUT_FT = VAL_OUT_FT + XADAPTER_SUFFIX
if XADAPTER_SUFFIX and not TEST_OUT_FT.endswith(XADAPTER_SUFFIX):
    TEST_OUT_FT = TEST_OUT_FT + XADAPTER_SUFFIX

if not MIN_FREE_GB:
    if PRECISION == "bf16":
        MIN_FREE_GB = 60
    elif VRAM_GB < 26:
        # L4 24GB에서는 여유 하한을 20으로 두면 시작조차 거부됩니다.
        MIN_FREE_GB = 16
    else:
        MIN_FREE_GB = 20

print()
print("GPU              : %s / VRAM %.1fGB" % (GPU_NAME, VRAM_GB))
print("모델             : %s (revision %s)" % (MODEL_ID, MODEL_REVISION))
print("정밀도 정책      : %s -> %s" % (PRECISION_POLICY, PRECISION))
print("적용 중인 config : %s" % CONFIG)
print("해상도           : min_pixels=%d, max_pixels=%d%s (시각 토큰 수는 8번 절 pixel_limits가 실측합니다)"
      % (MIN_PIXELS, MAX_PIXELS, " [작은 이미지 확대본]" if UPSCALE else ""))
print("프롬프트         : %s" % PROMPT_ID)
print("학습 범위        : %s (%s)" % (TRAIN_SCOPE, SPLIT_FILE))
print("base run-id      : %s" % BASE_RUN)
print("ft run-id        : %s" % FT_RUN)
if ADAPTER_RUN_ID:
    print("adapter 출처     : %s (다른 run에서 빌려 옵니다. 결과 이름에 %s 가 붙습니다)"
          % (ADAPTER_PATH, XADAPTER_SUFFIX))
else:
    print("adapter 출처     : %s (이 노트북이 학습하는 run입니다)" % ADAPTER_PATH)
print("프롬프트 비교 run-id : %s (선택 단계에서만 사용합니다)" % CMP_RUN)
print("해상도 비교 run-id   : %s (선택 단계에서만 사용합니다)" % RES_CMP_RUN)
print("학습 허용        : %s" % TRAINING_ALLOWED)
print("GPU 여유 하한    : %s GB" % MIN_FREE_GB)
print("가중치 캐시      : %s" % os.environ["VQA_MODEL_ROOT"])

_choice_path = RUN_ROOT_P / "colab_choice.json"
if _choice_path.exists():
    _choice = json.loads(_choice_path.read_text(encoding="utf-8"))
    print()
    print("참고: 20번 노트북은 VRAM %s GB에서 %s 를 골랐습니다 (정책 %s)."
          % (_choice.get("vram_gb"), _choice.get("config"), _choice.get("policy", "미기록")))
    print("이 값은 참고로만 출력하며, 지금 실행에는 위에서 정한 값을 씁니다.")

20번 노트북이 쓴 가중치 캐시를 그대로 사용합니다: /content/drive/MyDrive/vqa/hf_cache
캐시 여유 공간 : 178.6GB (27B 원본 스냅샷은 약 54GB입니다)
파라미터 셀의 CONFIG를 그대로 사용합니다: configs/colab_qwen35_27b_bf16_upscale.yaml

GPU              : NVIDIA A100-SXM4-80GB / VRAM 80.0GB
모델             : Qwen/Qwen3.5-27B (revision fc05daec18b0a78c049392ed2e771dde82bdf654)
정밀도 정책      : bf16 -> bf16
적용 중인 config : configs/colab_qwen35_27b_bf16_upscale.yaml
해상도           : min_pixels=1048576, max_pixels=2097152 [작은 이미지 확대본] (시각 토큰 수는 8번 절 pixel_limits가 실측합니다)
프롬프트         : v2_read_text
학습 범위        : full (artifacts/data/split_full.json)
base run-id      : colab27b_base_bf16_v2_read_text_upscale
ft run-id        : colab27b_ft_bf16_full_v2_read_text_upscale
adapter 출처     : /content/drive/MyDrive/vqa/runs/colab27b_ft_bf16_full_v2_read_text_upscale/final_adapter (이 노트북이 학습하는 run입니다)
프롬프트 비교 run-id : colab27b_base_bf16_v2_read_text_upscale (선택 단계에서만 사용합니다)
해상도 비교 run-id   : colab27b_base_bf16_v2_read_text_upscale (선택 단계에서만 사용합니다)
학습 허용       

## 5-1. 가중치 확보

20번 노트북을 돌리지 않았거나, 20번을 돌린 계정과 다른 계정으로 접속했다면 캐시에 가중치가
없습니다. 그대로 두면 9번 절의 추론 job이 시작하자마자 약 54GB를 내려받게 되고, 그 도중에
job이 재시작되면 받던 파일을 처음부터 다시 받는 일이 생깁니다. 그래서 이 절에서 먼저
내려받습니다. 이미 받아 둔 스냅샷이 있으면 아무 일도 하지 않습니다.

`nf4` 로 돌리더라도 내려받는 파일은 같습니다. 양자화는 원본 bf16 스냅샷을 읽어서 메모리에
올리는 시점에 수행하므로, 캐시에는 언제나 원본 크기인 약 54GB가 필요합니다.

내려받는 모델은 5번 절이 `CONFIG` 에서 읽은 `MODEL_ID` 와 `MODEL_REVISION` 입니다. 그래서
`CONFIG` 를 `QWEN38_UPSCALE_CONFIG` 로 바꾸면 이 절이 Qwen3.8-27B 스냅샷을 받습니다. 두 모델을
번갈아 돌리면 캐시에 54GB짜리 스냅샷이 두 벌, 모두 약 108GB가 쌓입니다. Drive 여유가 그만큼
없다면 파라미터 셀의 `MODEL_CACHE` 를 `"local"` 로 두어 가상 머신 로컬 디스크에 받거나, 다 쓴
쪽의 `models--Qwen--Qwen3.5-27B` 폴더를 지우고 한 벌만 두십시오.

Qwen3.5와 Qwen3.8 모두 접근 승인이 필요한 모델이 아니므로 토큰이 없어도 받을 수 있습니다.
다만 같은 주소에서 여러 번 받으면 익명 요청의 속도 제한에 걸릴 수 있습니다. Hugging Face에서 읽기
권한 토큰을 만들어 Colab 비밀 `HF_TOKEN` 으로 등록해 두면 이 셀이 그 값을 사용합니다.

In [7]:
# Hugging Face 토큰이 등록되어 있으면 씁니다. 없어도 Qwen3.5는 받을 수 있습니다.
try:
    from google.colab import userdata as _userdata

    _hf_token = _userdata.get("HF_TOKEN")
except Exception:
    # SecretNotFoundError, NotebookAccessError, ImportError를 모두 여기에서 받습니다.
    _hf_token = None
if _hf_token:
    os.environ["HF_TOKEN"] = _hf_token
    print("Colab 비밀 HF_TOKEN을 찾아서 환경 변수에 넣었습니다.")
    del _hf_token
else:
    print("HF_TOKEN이 없어 익명으로 내려받습니다. 속도 제한 오류가 나면 토큰을 등록하십시오.")

# 스냅샷 폴더 이름은 커밋 해시입니다. MODEL_REVISION 이 "main" 이면 그런 폴더가 없으므로
# 아래 검사가 언제나 어긋나고 내려받기를 한 번 더 시도합니다. 이미 받아 둔 파일은 그대로
# 재사용하므로 다시 받지는 않지만, 설정 파일에 커밋 해시를 적어 두는 편이 빠릅니다.
print("내려받을 모델 :", MODEL_ID, "(revision %s)" % MODEL_REVISION)
SNAPSHOT_DIR = (MODEL_ROOT_P / ("models--" + MODEL_ID.replace("/", "--"))
                / "snapshots" / MODEL_REVISION)
print("기대하는 스냅샷 위치:", SNAPSHOT_DIR)

if SNAPSHOT_DIR.is_dir():
    print("가중치가 이미 캐시에 있습니다. 내려받지 않습니다.")
elif not DOWNLOAD_WEIGHTS:
    print("DOWNLOAD_WEIGHTS 가 False이므로 내려받지 않습니다.")
    print("가중치가 없으면 9번 절의 추론 job이 시작하면서 직접 내려받습니다.")
else:
    if CACHE_FREE_GB < MODEL_CACHE_NEED_GB:
        raise RuntimeError(
            "캐시 위치 %s 의 여유가 %.1fGB뿐이라 약 54GB짜리 스냅샷이 들어가지 않습니다. "
            "파라미터 셀의 MODEL_CACHE 를 \"local\" 로 바꿔서 가상 머신 로컬 디스크에 받거나, "
            "Drive를 정리한 다음 5번 절부터 다시 실행하십시오."
            % (MODEL_ROOT, CACHE_FREE_GB)
        )
    _hf_bin = shutil.which("hf") or shutil.which("huggingface-cli")
    if _hf_bin:
        _dl = run([_hf_bin, "download", MODEL_ID, "--revision", MODEL_REVISION,
                   "--cache-dir", str(MODEL_ROOT_P)], check=False)
    else:
        print("hf 명령을 찾지 못해 python 모듈로 받습니다.")
        os.environ["VQA_MODEL_ID"] = MODEL_ID
        os.environ["VQA_MODEL_REV"] = MODEL_REVISION
        _dl = run([PY, "-c",
                   "import os, huggingface_hub as h;"
                   "p=h.snapshot_download(os.environ['VQA_MODEL_ID'],"
                   " revision=os.environ['VQA_MODEL_REV'],"
                   " cache_dir=os.environ['HF_HUB_CACHE']);"
                   "print('snapshot:', p)"], check=False)
    if _dl["rc"] != 0:
        raise RuntimeError(
            "가중치를 내려받지 못했습니다. 위 출력에서 원인을 확인하십시오. 이미 받은 부분은 "
            "캐시에 남아 있으므로 이 셀을 다시 실행하면 이어서 받습니다."
        )
    (RUN_ROOT_P / "model_snapshot.json").write_text(
        json.dumps({"model_id": MODEL_ID, "revision": MODEL_REVISION,
                    "cache_dir": str(MODEL_ROOT_P),
                    "download_seconds": round(_dl["seconds"], 1)},
                   ensure_ascii=False, indent=2), encoding="utf-8")
    print("내려받기를 %.1f분 만에 마쳤습니다." % (_dl["seconds"] / 60.0))

print("캐시 여유 공간 : %.1fGB" % _free_gb(MODEL_ROOT))

HF_TOKEN이 없어 익명으로 내려받습니다. 속도 제한 오류가 나면 토큰을 등록하십시오.
내려받을 모델 : Qwen/Qwen3.5-27B (revision fc05daec18b0a78c049392ed2e771dde82bdf654)
기대하는 스냅샷 위치: /content/drive/MyDrive/vqa/hf_cache/models--Qwen--Qwen3.5-27B/snapshots/fc05daec18b0a78c049392ed2e771dde82bdf654
가중치가 이미 캐시에 있습니다. 내려받지 않습니다.
캐시 여유 공간 : 178.6GB


## 6. 데이터 준비

압축 파일을 로컬 디스크에 풉니다. 이미 풀려 있으면 아무 일도 하지 않습니다. 압축 파일에
암호가 없다면 파라미터 셀의 `ARCHIVE_PASSWORD` 에 `none` 이라고 적으십시오.

In [8]:
# --- 데이터를 로컬 디스크에 준비합니다 ---
import getpass, shutil

data_root = Path(DATA_ROOT)


def locate_data_root(base):
    """압축 안에 최상위 폴더가 하나 있었다면 그 하위 폴더를 데이터 위치로 삼습니다."""
    if (base / "train.csv").exists():
        return base
    if base.is_dir():
        subs = [p for p in sorted(base.iterdir())
                if p.is_dir() and p.name != "__MACOSX" and (p / "train.csv").exists()]
        if len(subs) == 1:
            return subs[0]
    return base


def ensure_7z():
    """7z 명령이 없으면 조용히 설치합니다."""
    if shutil.which("7z"):
        return
    print("7z 명령이 없어 p7zip-full을 설치합니다. 30초 정도 걸립니다.")
    run(["apt-get", "-qq", "install", "-y", "p7zip-full"], check=False, quiet=True)
    if not shutil.which("7z"):
        raise RuntimeError("p7zip-full 설치에 실패했습니다. 런타임의 네트워크 상태를 확인하십시오.")


data_root.mkdir(parents=True, exist_ok=True)
found_root = locate_data_root(data_root)

if (found_root / "train.csv").exists() and (found_root / "train").is_dir():
    print("이미 데이터가 준비되어 있습니다:", found_root)
else:
    ensure_7z()
    if str(ARCHIVE_PASSWORD).strip().lower() == "none":
        pwd = None
        print("ARCHIVE_PASSWORD가 none이므로 암호가 없는 압축 파일로 처리합니다.")
    else:
        pwd = ARCHIVE_PASSWORD or getpass.getpass("압축 파일 비밀번호를 입력하십시오: ")
    cmd = ["7z", "x", "-y"]
    if pwd:
        cmd.append("-p" + pwd)
    cmd += ["-o" + str(data_root), ARCHIVE_PATH]
    print("$ 7z x -y %s -o%s %s" % ("-p***" if pwd else "", data_root, ARCHIVE_PATH))
    run(cmd, echo=False)
    found_root = locate_data_root(data_root)

if found_root != data_root:
    print("압축 안의 하위 폴더를 데이터 위치로 사용합니다:", found_root)

data_root = found_root
DATA_ROOT = str(data_root)
DATA_ROOT_P = data_root
os.environ["VQA_DATA_ROOT"] = str(data_root)


for name in ("train", "test", "dev"):
    p = data_root / name
    print("%-8s 이미지 %d장" % (name, len(list(p.glob("*.jpg"))) if p.is_dir() else 0))
print("VQA_DATA_ROOT:", os.environ["VQA_DATA_ROOT"])

압축 파일 비밀번호를 입력하십시오: ··········
$ 7z x -y -p*** -o/content/data /content/drive/MyDrive/ssafy-16-2-ai.zip

7-Zip 23.01 (x64) : Copyright (c) 1999-2023 Igor Pavlov : 2023-06-20
 64-bit locale=en_US.UTF-8 Threads:12 OPEN_MAX:1048576

Scanning the drive for archives:
1 file, 1799380978 bytes (1717 MiB)

Extracting archive: /content/drive/MyDrive/ssafy-16-2-ai.zip
--
Path = /content/drive/MyDrive/ssafy-16-2-ai.zip
Type = zip
Physical Size = 1799380978

Everything is Ok

Folders: 3
Files: 16118
Size:       1801552403
Compressed: 1799380978
[rc=0] 34.5초
train    이미지 6714장
test     이미지 6714장
dev      이미지 2683장
VQA_DATA_ROOT: /content/data


## 7. 실행 도우미

아래 셀은 이 노트북이 쓰는 도우미 함수를 정의합니다. 직접 고칠 일은 없습니다.

`gpu_step` 은 GPU를 쓰는 작업 하나를 띄우고 끝날 때까지 기다립니다. `vqa.jobs start` 는 같은
run-root에 살아 있는 job이 있으면 시작을 거부하고, `--min-free-gb` 로 지정한 만큼 GPU 여유
메모리가 없어도 거부합니다. 이 두 가지가 작업이 겹쳐서 out of memory가 나는 상황을 막아 줍니다.

모든 job은 `--auto-restart` 와 함께 띄웁니다. 이 옵션이 없으면 종료 코드를 기록해 줄 감시
프로세스가 붙지 않고, 그러면 `vqa.jobs wait` 가 작업의 성공과 실패를 구분하지 못한 채 언제나
0을 돌려줍니다. 실패한 작업을 성공으로 잘못 읽으면 그 뒤의 단계가 모두 어긋나므로, 아래 셀은
`AUTO_RESTART` 가 0이면 그 자리에서 멈춥니다. `gpu_step` 에 `auto_restart` 인자를 주면 그
단계에서만 재시작 횟수를 다르게 정할 수 있습니다. 같은 이유로 1보다 작은 값은 받지 않습니다.

`post_train_ready` 는 15번 절부터 17번 절까지의 셀이 학습과 겹치지 않게 막아 줍니다. 학습
job이 아직 살아 있거나 `final_adapter` 가 아직 없으면, 이 함수가 안내만 출력하고 거짓을
돌려주므로 해당 셀은 아무 일도 하지 않고 끝납니다. 그래서 메뉴의 "모두 실행"을 눌러도 학습을
띄운 직후에 추론이 따라 들어가지 않습니다.

`OVERRIDES` 에는 파라미터 셀의 `PROMPT_ID` 가 항상 들어갑니다. 11번 노트북이 동결한
`artifacts/input_config_frozen.json` 이 있으면 거기에서 해상도만 함께 가져옵니다. 동결 파일에
적힌 프롬프트는 쓰지 않습니다. 이 노트북에서 무엇으로 실행했는지가 파라미터 셀 하나만 보면
드러나야 하기 때문입니다.

In [9]:
# --- 명령에 공통으로 붙는 인자 ---
# --auto-restart 없이 띄운 job에는 종료 코드를 기록해 줄 감시 프로세스가 붙지 않습니다. 그러면
# vqa.jobs wait 가 성공과 실패를 구분하지 못하고 언제나 0을 돌려주므로, 실패를 성공으로 잘못
# 읽게 됩니다. 그래서 이 노트북은 모든 job을 --auto-restart 와 함께 띄웁니다.
if not AUTO_RESTART:
    raise RuntimeError(
        "AUTO_RESTART 가 0입니다. 감시 프로세스가 붙지 않아 vqa.jobs wait 가 작업의 성공과 "
        "실패를 구분하지 못합니다. 파라미터 셀에서 AUTO_RESTART 를 1 이상으로 고치고 "
        "(권장값은 3입니다) 이 셀을 다시 실행하십시오."
    )
JOB_EXTRA = ["--auto-restart", str(AUTO_RESTART)]
ALSO_BASE_ARGS = ["--also-base"] if ALSO_BASE else []

OVERRIDES = ["--set", "prompt_id=%s" % PROMPT_ID] + PRECISION_OVERRIDES
_frozen_path = Path("artifacts/input_config_frozen.json")
if _frozen_path.exists():
    _frozen = json.loads(_frozen_path.read_text(encoding="utf-8"))
    _mp = (_frozen.get("max_pixels") or {}).get("colab_qwen35_27b")
    if _mp and UPSCALE:
        # 확대본을 고른 실행에서 동결 값을 덮어쓰면 확대한 이미지가 다시 잘립니다.
        print("확대본을 쓰므로 동결 파일의 max_pixels=%d 는 적용하지 않습니다." % _mp)
    elif _mp:
        OVERRIDES += ["--set", "max_pixels=%d" % _mp]
        print("동결 파일에서 max_pixels=%d 를 가져왔습니다." % _mp)
    if _frozen.get("prompt_id") and _frozen["prompt_id"] != PROMPT_ID:
        print("참고: 동결 파일의 프롬프트는 %s 이지만 이번 실행은 %s 를 씁니다."
              % (_frozen["prompt_id"], PROMPT_ID))


def overrides_for(prompt_id):
    """프롬프트만 바꾼 override 목록을 만듭니다. 비교 단계에서 씁니다."""
    out = list(OVERRIDES)
    for index, item in enumerate(out):
        if isinstance(item, str) and item.startswith("prompt_id="):
            out[index] = "prompt_id=%s" % prompt_id
    return out


def drop_override(args, key):
    """`--set <key>=...` 쌍을 목록에서 걷어냅니다.

    설정 파일이 다른 실행에 같은 override를 그대로 넘기면 그 파일에 적힌 값이 덮여 버립니다.
    해상도 비교에서 확대본의 max_pixels 와 precision 이 바로 그런 값입니다.
    """
    out = []
    index = 0
    while index < len(args):
        item = args[index]
        if (item == "--set" and index + 1 < len(args)
                and str(args[index + 1]).startswith(key + "=")):
            index += 2
            continue
        out.append(item)
        index += 1
    return out


def pred_file(run_id, out_name):
    """예측 결과 CSV의 경로입니다."""
    return RUN_ROOT_P / run_id / "preds" / (out_name + ".csv")


def pred_done(run_id, out_name):
    """예측이 끝까지 완료되었는지 봅니다. manifest는 마지막에만 기록됩니다."""
    return Path(str(pred_file(run_id, out_name)) + ".manifest.json").exists()


def job_status(name):
    res = run([PY, "-m", "vqa.jobs", "status", "--name", name],
              check=False, quiet=True, hint=False, echo=False)
    out = res["output"].strip()
    if res["rc"] == 0:
        print(out)
    elif "기록 없음" in out:
        print("job %s: 아직 시작하지 않았습니다." % name)
    else:
        print(out)
    return res


def job_alive(name):
    """job이 아직 돌고 있는지 봅니다. 감시 프로세스와 실제 작업 프로세스를 모두 확인합니다."""
    res = run([PY, "-m", "vqa.jobs", "status", "--name", name, "--json"],
              check=False, quiet=True, hint=False, echo=False)
    text = res["output"].strip()
    if not text:
        return False
    try:
        status = json.loads(text)
    except ValueError:
        return False
    return bool(status.get("any_alive"))


# 추론에 쓰는 adapter의 위치입니다. 5번 절에서 ADAPTER_RUN_ID 를 보고 정해 둔 값을 그대로
# 씁니다. 학습 전에도 이름만은 정의되어 있어야 뒤의 셀들이 NameError 없이 안내만 출력하고
# 넘어갈 수 있습니다.
ADAPTER = ADAPTER_PATH


def post_train_ready():
    """학습 이후 단계를 지금 실행해도 되는지 판정하고, 되지 않으면 이유를 설명합니다.

    메뉴의 "모두 실행"을 누르면 학습을 띄운 직후에 아래쪽 셀들이 곧바로 이어서 실행됩니다.
    그대로 두면 학습과 추론이 GPU 한 장을 함께 쓰게 되므로, 학습 job이 아직 살아 있거나
    final_adapter가 아직 만들어지지 않았으면 아무 일도 하지 않고 안내만 출력합니다.
    """
    train_job = FT_RUN + "_train"
    if job_alive(train_job):
        print("학습 job %s 이 아직 돌고 있으므로 이 셀은 건너뜁니다." % train_job)
        print("14번 절의 상태 확인 셀로 남은 시간을 확인하시고, 학습이 끝난 다음에 이 셀을")
        print("다시 실행하십시오. 지금 실행하면 GPU를 함께 쓰게 되어 out of memory가 납니다.")
        return False
    if not Path(ADAPTER).is_dir():
        print("학습된 adapter가 아직 없으므로 이 셀은 건너뜁니다: %s" % ADAPTER)
        print("13번 절에서 학습을 시작하고, 학습이 끝난 뒤에 이 셀을 다시 실행하십시오.")
        return False
    return True


def fold0_eval_skipped():
    """fold 0 평가를 건너뛰어야 하면 이유를 출력하고 참을 돌려줍니다.

    TRAIN_SCOPE 가 "full" 이면 fold 0의 1,344문항도 학습에 들어갑니다. 그 모델의 fold 0
    정확도는 학습 데이터를 다시 맞힌 점수이므로 일반화 성능을 나타내지 않고, 원본과의
    McNemar 비교도 같은 이유로 뜻을 잃습니다.
    """
    if TRAIN_SCOPE != "full":
        return False
    print("TRAIN_SCOPE 가 \"full\" 이므로 이 셀을 건너뜁니다.")
    print("전체 데이터로 학습한 모델은 fold 0 문항도 학습에 썼습니다. 그래서 fold 0 정확도는")
    print("학습 데이터를 다시 맞힌 점수이며, 일반화 성능도 학습 효과도 나타내지 못합니다.")
    print("설정을 검증한 fold 0 비교는 TRAIN_SCOPE 가 \"fold0\" 이던 run에 이미 남아 있습니다.")
    print("이 실행에서는 17번 절의 test 추론과 제출 파일 생성만 진행하십시오.")
    return True


def gpu_step(name, cmd, *, done=False, wait=True, min_free_gb=None, poll=None, auto_restart=None):
    """GPU 작업 하나를 띄우고 끝날 때까지 기다립니다.

    done이 True이면 이미 끝난 작업이므로 모델을 올리지 않고 건너뜁니다.
    wait가 False이면 띄우기만 합니다. 학습에만 씁니다.
    auto_restart를 주면 이 단계에서만 재시작 횟수를 AUTO_RESTART 대신 그 값으로 씁니다.
    실패가 결정론적인 단계에서 27B 모델을 여러 번 다시 올리지 않기 위해서입니다.
    0은 받지 않습니다. 감시 프로세스가 붙지 않으면 wait가 실패를 성공으로 돌려주기 때문입니다.
    """
    if done:
        print("이미 끝나 있으므로 건너뜁니다: %s" % name)
        return "skipped"

    if auto_restart is None:
        extra = JOB_EXTRA
    elif int(auto_restart) < 1:
        raise ValueError(
            "auto_restart 는 1 이상이어야 합니다. 0이면 감시 프로세스가 붙지 않아 "
            "vqa.jobs wait 가 작업의 성공과 실패를 구분하지 못합니다."
        )
    else:
        extra = ["--auto-restart", str(int(auto_restart))]

    args = [PY, "-m", "vqa.jobs", "start", "--name", name] + extra
    free_gb = MIN_FREE_GB if min_free_gb is None else min_free_gb
    if free_gb:
        args += ["--min-free-gb", str(free_gb)]
    args += ["--"] + [str(c) for c in cmd]

    started = run(args, check=False)
    if started["rc"] != 0:
        raise RuntimeError(
            "job %s 을 시작하지 못했습니다. 위 출력을 확인하십시오. 다른 job이 아직 돌고 있으면 "
            "그 job이 끝난 뒤에 이 셀을 다시 실행하면 됩니다. GPU 여유 메모리가 모자란다는 "
            "메시지가 보이면 런타임을 재시작한 다음 4번 절부터 다시 실행하십시오." % name
        )
    if not wait:
        print("job %s 을 띄웠습니다. 이 셀은 기다리지 않고 끝납니다." % name)
        return "started"

    print("job %s 이 끝날 때까지 기다립니다. 이 셀을 멈춰도 job은 계속 돌아갑니다." % name)
    waited = run([PY, "-m", "vqa.jobs", "wait", "--name", name,
                  "--poll", str(poll or JOB_POLL_SEC)], check=False)
    if waited["rc"] == 0:
        print("job %s 이 정상으로 끝났습니다." % name)
        return "done"
    if waited["rc"] == 2:
        raise RuntimeError(
            "job %s 을 기다리는 동안 제한 시간을 넘겼습니다. job은 아직 돌고 있을 수 있으므로 "
            "`python -m vqa.jobs status --name %s` 로 확인하십시오." % (name, name)
        )
    raise RuntimeError(
        "job %s 이 실패로 끝났습니다. `python -m vqa.jobs tail --name %s --lines 40` 으로 "
        "마지막 로그를 읽고 원인을 확인하십시오." % (name, name)
    )


def evaluate(pred_path, out_prefix, note):
    """예측 파일을 fold 0 정답과 비교합니다. GPU를 쓰지 않습니다."""
    Path("artifacts/eval").mkdir(parents=True, exist_ok=True)
    return script("evaluate_run.py",
                  "--pred", str(pred_path),
                  "--gold", str(DATA_ROOT_P / "train.csv"),
                  "--val-ids", VAL_IDS_FILE,
                  "--wrong-out", "artifacts/eval/%s_wrong.csv" % out_prefix,
                  "--json-out", "artifacts/eval/%s.json" % out_prefix,
                  "--note", note,
                  check=False)


def make_submission(pred_path, out_path):
    """예측 파일에서 제출 파일을 만들고 검사기까지 돌립니다."""
    Path(out_path).parent.mkdir(parents=True, exist_ok=True)
    sample = str(DATA_ROOT_P / "sample_submission.csv")
    vqa("submit", "--pred", str(pred_path), "--sample", sample, "--out", str(out_path))
    script("check_submission.py", "--submission", str(out_path), "--sample", sample)
    print()
    print("제출 후보: %s" % out_path)
    print("sha256   : %s" % sha256_of(out_path))


print("config    :", CONFIG)
print("job 옵션  :", JOB_EXTRA or "(없음)")
print("입력 설정 :", OVERRIDES)

config    : configs/colab_qwen35_27b_bf16_upscale.yaml
job 옵션  : ['--auto-restart', '3']
입력 설정 : ['--set', 'prompt_id=v2_read_text']


## 8. preflight

본 학습에 들어가기 전에 토큰과 템플릿, label 위치, 해상도 적용, 커널 경로를 한 번에
점검합니다. 이 점검이 걸러 내는 것은 몇 시간 뒤에야 드러나는 종류의 실패입니다.

결과에서 특히 두 항목을 확인하십시오.

- `pixel_limits` 는 설정한 `min_pixels` 와 `max_pixels` 가 processor에 실제로 적용되었는지를
  가장 큰 이미지와 가장 작은 이미지로 직접 확인합니다. 시각 토큰 수가 기대 상한을 넘으면
  FAIL로 표시됩니다.
- `forward_backward` 의 `peak_vram_gb` 는 학습이 실제로 쓸 메모리의 근사값입니다. 이 검사가
  OOM을 보려고 고르는 표본은 원래 질문과 선택지의 글자 수가 가장 긴 문항이었습니다. 그런데
  메모리를 지배하는 쪽은 글자가 아니라 시각 토큰이므로, 해상도를 올린 설정에서는 가장 큰
  이미지가 표본에 한 장도 들어가지 않을 수 있습니다. 그러면 preflight를 통과하고도 본 학습
  도중에 out of memory가 납니다. 그래서 이 노트북은 `--largest-image-samples` 로 원본 픽셀
  수가 가장 큰 이미지 `PREFLIGHT_LARGEST_IMAGES` 장을 OOM 표본에 강제로 넣습니다. 어떤
  문항이 들어갔는지는 요약의 `largest_image_samples` 에 나옵니다.
- `kernel_fast_path` 는 Gated DeltaNet이 가속 커널을 쓰는지 봅니다. 참조 구현으로 내려갔으면
  WARN으로 표시하고 설치 명령을 함께 알려 줍니다. 학습 자체는 가능하므로 종료 코드는 0입니다.
  다만 학습 시간이 약 두 배가 되므로 그대로 진행할지 먼저 판단하십시오.

preflight도 분리 프로세스로 뜨기 때문에 그 출력은 모두 로그 파일에 쌓입니다. 그래서 아래 셀은
작업이 끝난 다음에 결과 JSON을 읽어, 검사별 PASS/WARN/FAIL과 속도 측정값을 이 자리에 다시
출력합니다. 검사가 실패한 경우에도 요약을 먼저 출력하고 그 다음에 오류를 알립니다. 또한
preflight의 실패는 대부분 다시 실행해도 같은 자리에서 반복되므로, 이 단계에서만 재시작 횟수를
1로 낮춰 두었습니다.

In [10]:
# preflight의 FAIL은 대개 결정론적이라 다시 띄워도 같은 자리에서 실패합니다. 27B 모델을 여러
# 번 올리며 수십 분을 버리지 않도록 이 단계만 재시작 횟수를 1로 낮춥니다.
PREFLIGHT_RESTARTS = 1
PF_JOB = "preflight_" + PRECISION


def preflight_results():
    """preflight가 남긴 결과 JSON 목록입니다. 파일 이름이 타임스탬프라 정렬하면 시간순입니다."""
    pf_dir = RUN_ROOT_P / "preflight"
    return sorted(pf_dir.glob("*.json")) if pf_dir.is_dir() else []


def show_preflight_summary(before):
    """preflight의 출력은 job 로그로 가므로, 결과 JSON을 읽어 요약을 이 셀에 다시 찍습니다."""
    fresh = [p for p in preflight_results() if p not in before]
    if not fresh:
        print()
        print("이번 실행의 preflight 결과 JSON을 찾지 못했습니다. 결과를 쓰기 전에 끝났을 수 있습니다.")
        print("`python -m vqa.jobs tail --name %s --lines 80` 으로 로그를 직접 확인하십시오." % PF_JOB)
        return
    path = fresh[-1]
    data = json.loads(path.read_text(encoding="utf-8"))
    results = data.get("results") or []
    counts = {}
    for item in results:
        counts[item.get("status")] = counts.get(item.get("status"), 0) + 1
    print()
    print("=" * 68)
    print("PREFLIGHT %s  (%s)"
          % (data.get("overall"), "  ".join("%s=%d" % kv for kv in sorted(counts.items()))))
    print("=" * 68)
    for item in results:
        line = "  %-4s %s" % (item.get("status"), item.get("name"))
        if item.get("reason"):
            line += " - %s" % item["reason"]
        print(line)

    def show_detail(check_name, title, keys):
        for item in results:
            if item.get("name") != check_name:
                continue
            detail = item.get("detail") or {}
            picked = [(k, detail[k]) for k in keys if k in detail]
            if not picked:
                continue
            print()
            print(title)
            for key, value in picked:
                print("  %-24s : %s" % (key, value))

    show_detail("forward_backward", "학습 속도 (microbatch 1개 기준)",
                ("sec_per_microbatch_mean", "sec_prepare_mean", "sec_compute_mean",
                 "sec_per_optimizer_step", "peak_vram_gb", "peak_vram_gb_longest",
                 "largest_image_samples"))
    show_detail("inference", "추론 속도 (문항 1개 기준)",
                ("seconds_per_item", "sec_prepare_mean", "sec_compute_mean"))
    print()
    print("기록: %s" % path)


# 학습을 허용하지 않는 가상 머신에서는 학습 점검을 생략합니다. forward+backward는 역전파
# 활성값까지 올리므로, 추론만 하려는 24GB급 GPU에서는 이 점검 하나 때문에 out of memory가
# 납니다. --n-train-steps 를 0으로 주면 preflight가 forward_backward를 SKIP으로 남깁니다.
PF_TRAIN_STEPS = PREFLIGHT_TRAIN_STEPS if TRAINING_ALLOWED else 0
if not PF_TRAIN_STEPS:
    print("학습을 허용하지 않는 가상 머신이므로 preflight의 학습 점검을 생략하고 추론만 봅니다.")

cmd = [PY, "-m", "vqa.preflight",
       "--config", CONFIG,
       "--n-train-steps", str(PF_TRAIN_STEPS),
       "--n-infer", str(PREFLIGHT_INFER_ITEMS),
       "--largest-image-samples", str(PREFLIGHT_LARGEST_IMAGES)] + OVERRIDES
_pf_before = preflight_results()
try:
    gpu_step(PF_JOB, cmd, auto_restart=PREFLIGHT_RESTARTS)
finally:
    # 실패했더라도 어느 검사가 걸렸는지 먼저 보이게 하고, 그 다음에 예외가 올라가게 합니다.
    show_preflight_summary(_pf_before)

$ /usr/bin/python3 -m vqa.jobs start --name preflight_bf16 --auto-restart 1 --min-free-gb 60 -- /usr/bin/python3 -m vqa.preflight --config configs/colab_qwen35_27b_bf16_upscale.yaml --n-train-steps 8 --n-infer 20 --largest-image-samples 2 --set prompt_id=v2_read_text
[jobs] 시작: preflight_bf16 pid=9298 log=/content/drive/MyDrive/vqa/runs/jobs/preflight_bf16.log
[rc=0] 12.3초
job preflight_bf16 이 끝날 때까지 기다립니다. 이 셀을 멈춰도 job은 계속 돌아갑니다.
$ /usr/bin/python3 -m vqa.jobs wait --name preflight_bf16 --poll 30
[jobs] preflight_bf16 대기 중 pid=9298 child_pid=9300 | [jobs] supervise 시작 attempt=0: /usr/bin/python3 -m vqa.preflight --config configs/colab_qwen35_27b_bf16_upscale.yaml --n-train-steps 8 --n-infer 20 --largest-image-samples 2 --set prompt_id=v2_read_text
[jobs] preflight_bf16 대기 중 pid=9298 child_pid=9300 | [WARN] pixel_limits — 최대 1700x720 -> 시각 토큰 1166개, 최소 720x720 -> 시각 토큰 1024개 (기대 상한 2048개, 기대 하한 1024개). 다만 표본에서 가장 큰 이미지도 원본이 max_pixels보다 작아서 상한이 실제로 걸리는지는 확인하지 못했습니다.
[jobs] preflight_bf

## 9. fold 0을 원본 27B로 추론하고 평가합니다

학습하지 않은 27B로 팀 공식 검증 분할을 추론합니다. 결합 가중치를 고를 때 쓰는 기준 확률이고,
학습이 실제로 도움이 되었는지 판단할 때의 비교 대상이기도 하므로 반드시 먼저 확보해야 합니다.

진행 상황을 따로 보고 싶으면 다른 셀에서 `python -m vqa.jobs tail --name <job 이름> --lines 20`
을 실행하십시오. Drive에서 가중치를 읽어 오는 데 수 분이 걸리므로 첫 진행 로그가 나오기까지
시간이 걸립니다.

In [ ]:
cmd = [PY, "-m", "vqa.predict",
       "--config", CONFIG, "--run-id", BASE_RUN,
       "--csv", "train.csv", "--ids-file", VAL_IDS_FILE,
       "--adapter", "none", "--out-name", VAL_OUT_BASE] + OVERRIDES
gpu_step(BASE_RUN + "_val", cmd, done=pred_done(BASE_RUN, VAL_OUT_BASE))

In [ ]:
evaluate(pred_file(BASE_RUN, VAL_OUT_BASE), BASE_RUN + "_" + VAL_OUT_BASE,
         "27B 원본, 프롬프트 %s, 정밀도 %s" % (PROMPT_ID, PRECISION))

## 10. (선택) 프롬프트를 바꾸어 비교합니다

이 절은 선택입니다. `RUN_PROMPT_COMPARE` 가 `False` 이면 아무 일도 하지 않고 넘어갑니다.
fold 0 전체를 한 번 더 추론하므로 9번 절과 같은 만큼의 GPU 시간이 듭니다. compute unit 예산에
여유가 있을 때에만 `True` 로 바꾸십시오.

비교 결과는 `COMPARE_PROMPT_ID` 로 만든 별도의 run-id에 저장되므로 9번 절의 결과를 덮어쓰지
않습니다. 프롬프트 선택은 fold 0에서만 비교하며 test로는 어떤 조정도 하지 않습니다.

In [ ]:
if not RUN_PROMPT_COMPARE:
    print("RUN_PROMPT_COMPARE 가 False이므로 프롬프트 비교를 건너뜁니다.")
else:
    cmd = [PY, "-m", "vqa.predict",
           "--config", CONFIG, "--run-id", CMP_RUN,
           "--csv", "train.csv", "--ids-file", VAL_IDS_FILE,
           "--adapter", "none", "--out-name", VAL_OUT_BASE] + overrides_for(COMPARE_PROMPT_ID)
    gpu_step(CMP_RUN + "_val", cmd, done=pred_done(CMP_RUN, VAL_OUT_BASE))

In [ ]:
if not RUN_PROMPT_COMPARE:
    print("RUN_PROMPT_COMPARE 가 False이므로 프롬프트 비교 평가를 건너뜁니다.")
else:
    evaluate(pred_file(CMP_RUN, VAL_OUT_BASE), CMP_RUN + "_" + VAL_OUT_BASE,
             "27B 원본, 프롬프트 %s" % COMPARE_PROMPT_ID)
    script("compare_predictions.py",
           "--pred-a", str(pred_file(BASE_RUN, VAL_OUT_BASE)),
           "--pred-b", str(pred_file(CMP_RUN, VAL_OUT_BASE)),
           "--gold", str(DATA_ROOT_P / "train.csv"),
           "--val-ids", VAL_IDS_FILE,
           "--label-a", PROMPT_ID, "--label-b", COMPARE_PROMPT_ID,
           "--json-out", "artifacts/eval/prompt_compare.json",
           check=False)

## 10-1. (선택) 해상도를 바꾸어 비교합니다

이 절도 선택입니다. `RUN_RESOLUTION_COMPARE` 가 `False` 이면 아무 일도 하지 않고 넘어갑니다.
fold 0의 1,344문항을 한 번 더 추론하므로 9번 절과 같은 만큼의 GPU 시간이 듭니다. 확대본은 시각
토큰이 늘어나므로 실제로는 조금 더 걸립니다.

바꾸는 것은 해상도 하나뿐입니다. 프롬프트도 정밀도도 9번 절과 같게 두고 `min_pixels` 만 올린
`RESOLUTION_COMPARE_CONFIG` 로 추론합니다. 결과는 `_upscale` 이 붙은 별도의 run-id에 저장되므로
9번 절의 결과를 덮어쓰지 않습니다. 확대본이 원래 담고 있는 `max_pixels` 상향도 그대로 살려야
하므로, 이 절은 동결 파일에서 가져온 `max_pixels` 덮어쓰기를 걷어낸 다음에 실행합니다.

추론이 끝나면 두 가지를 확인하십시오. 먼저 `compare_predictions.py` 가 McNemar 검정으로 전체
1,344문항의 차이를 검정합니다. 그다음에 `ERROR_TYPES_CSV` 에서 판정이 `모델 오류: 작은 글자`
인 문항만 뽑아, 그 문항들이 기본 해상도와 확대본에서 각각 맞았는지 틀렸는지를 표로 출력합니다.
해상도를 올린 까닭이 바로 그 문항들이므로, 전체 정확도가 거의 그대로여도 이 표에서 개선이
보이면 방향은 맞다고 판단할 수 있습니다. 판정 파일이 없으면 이 표만 건너뜁니다.

In [ ]:
if not RUN_RESOLUTION_COMPARE:
    print("RUN_RESOLUTION_COMPARE 가 False이므로 해상도 비교를 건너뜁니다.")
elif RES_CMP_RUN == BASE_RUN:
    raise RuntimeError(
        "해상도 비교 설정 %s 가 9번 절의 설정과 같은 run-id를 만듭니다. 두 결과가 한 폴더에 "
        "섞이므로 실행하지 않습니다. 파라미터 셀의 CONFIG 와 RESOLUTION_COMPARE_CONFIG 를 "
        "서로 다른 해상도로 맞추십시오." % RESOLUTION_COMPARE_CONFIG
    )
else:
    # 확대본에 적힌 max_pixels 와 precision 을 덮어쓰지 않도록 두 값을 걷어낸 다음, 이 설정
    # 파일에 맞는 정밀도 덮어쓰기만 다시 붙입니다.
    RES_CMP_OVERRIDES = drop_override(drop_override(OVERRIDES, "max_pixels"), "precision")
    RES_CMP_OVERRIDES += precision_overrides_for(RESOLUTION_COMPARE_CONFIG)
    print("해상도 비교 설정 :", RESOLUTION_COMPARE_CONFIG)
    print("입력 설정        :", RES_CMP_OVERRIDES)
    cmd = [PY, "-m", "vqa.predict",
           "--config", RESOLUTION_COMPARE_CONFIG, "--run-id", RES_CMP_RUN,
           "--csv", "train.csv", "--ids-file", VAL_IDS_FILE,
           "--adapter", "none", "--out-name", VAL_OUT_BASE] + RES_CMP_OVERRIDES
    gpu_step(RES_CMP_RUN + "_val", cmd, done=pred_done(RES_CMP_RUN, VAL_OUT_BASE))

In [ ]:
if not RUN_RESOLUTION_COMPARE:
    print("RUN_RESOLUTION_COMPARE 가 False이므로 해상도 비교 평가를 건너뜁니다.")
elif not (pred_done(BASE_RUN, VAL_OUT_BASE) and pred_done(RES_CMP_RUN, VAL_OUT_BASE)):
    print("비교에 필요한 예측 두 개가 아직 모두 끝나지 않아 이 셀은 건너뜁니다.")
    print("  기본 해상도 fold 0 : %s" % ("완료" if pred_done(BASE_RUN, VAL_OUT_BASE) else "미완료"))
    print("  확대본 fold 0      : %s" % ("완료" if pred_done(RES_CMP_RUN, VAL_OUT_BASE) else "미완료"))
else:
    evaluate(pred_file(RES_CMP_RUN, VAL_OUT_BASE), RES_CMP_RUN + "_" + VAL_OUT_BASE,
             "27B 원본, 해상도 확대본, 정밀도 %s" % PRECISION)
    script("compare_predictions.py",
           "--pred-a", str(pred_file(BASE_RUN, VAL_OUT_BASE)),
           "--pred-b", str(pred_file(RES_CMP_RUN, VAL_OUT_BASE)),
           "--gold", str(DATA_ROOT_P / "train.csv"),
           "--val-ids", VAL_IDS_FILE,
           "--label-a", "base_res", "--label-b", "upscale",
           "--changed-out", "artifacts/eval/resolution_compare_changed.csv",
           "--json-out", "artifacts/eval/resolution_compare.json",
           check=False)

    # --- 작은 글자 때문에 틀렸다고 판정한 문항만 따로 봅니다 ---
    from vqa.evaluate import load_gold as _load_gold
    from vqa.predictions import decide_answers as _decide, load_predictions as _load_preds

    _err_path = Path(ERROR_TYPES_CSV)
    if not _err_path.exists():
        print()
        print("%s 가 없어서 작은 글자 문항 표는 건너뜁니다." % _err_path)
        print("이 파일은 fold 0 오답 점검 결과이며, 저장소의 artifacts/data 아래에 있습니다.")
    else:
        _small = [r["id"] for r in read_csv_rows(_err_path)
                  if (r.get("consensus") or "").strip() == SMALL_TEXT_CONSENSUS]
        _gold = _load_gold(str(DATA_ROOT_P / "train.csv")).set_index("id")["answer"].to_dict()
        _a = _decide(_load_preds(pred_file(BASE_RUN, VAL_OUT_BASE))).set_index("id")["answer"].to_dict()
        _b = _decide(_load_preds(pred_file(RES_CMP_RUN, VAL_OUT_BASE))).set_index("id")["answer"].to_dict()

        print()
        print("=" * 72)
        print("판정이 \"%s\" 인 %d문항의 정오 변화" % (SMALL_TEXT_CONSENSUS, len(_small)))
        print("=" * 72)
        print("  %-18s %6s %8s %8s  %s" % ("id", "정답", "기본", "확대본", "변화"))
        _counts = {"개선": 0, "악화": 0, "여전히 오답": 0, "계속 정답": 0, "예측 없음": 0}
        for _id in _small:
            _g = _gold.get(_id)
            _pa, _pb = _a.get(_id), _b.get(_id)
            if _g is None or _pa is None or _pb is None:
                _label = "예측 없음"
            else:
                _ok_a, _ok_b = (_pa == _g), (_pb == _g)
                if _ok_a and _ok_b:
                    _label = "계속 정답"
                elif _ok_b:
                    _label = "개선"
                elif _ok_a:
                    _label = "악화"
                else:
                    _label = "여전히 오답"
            _counts[_label] += 1
            print("  %-18s %6s %8s %8s  %s"
                  % (_id, _g or "?", _pa or "?", _pb or "?", _label))
        print("-" * 72)
        print("  " + "  ".join("%s %d건" % kv for kv in _counts.items() if kv[1]))
        print()
        print("개선과 악화의 건수를 함께 보십시오. 12문항은 표본이 작아서 개선 두세 건만으로는")
        print("해상도 상향이 효과가 있다고 단정할 수 없습니다. 전체 1,344문항의 McNemar 결과와")
        print("함께 판단하시고, 두 결과가 어긋나면 그 사실을 그대로 팀 채널에 공유하십시오.")

## 11. 전체 test를 원본 27B로 추론합니다

첫 유효 제출과 확률 캐시를 확보하는 단계입니다. 이 작업이 끝나야 수정이 제출을 준비할 수
있습니다. 문항이 6,714개이므로 fold 0보다 다섯 배쯤 오래 걸립니다.

In [ ]:
cmd = [PY, "-m", "vqa.predict",
       "--config", CONFIG, "--run-id", BASE_RUN,
       "--csv", "test.csv",
       "--adapter", "none", "--out-name", TEST_OUT_BASE] + OVERRIDES
gpu_step(BASE_RUN + "_test", cmd, done=pred_done(BASE_RUN, TEST_OUT_BASE))

## 12. 원본 모델의 제출 파일을 만들고 검사합니다

검사기를 통과하면 파일과 sha256을 수정에게 넘깁니다. GPU를 쓰지 않는 단계입니다.

In [ ]:
make_submission(pred_file(BASE_RUN, TEST_OUT_BASE),
                "artifacts/submission/%s_base.csv" % BASE_RUN)

## 13. 학습을 시작합니다

여기부터가 열 시간 안팎이 걸리는 단계입니다. 앞의 셀들과 달리 이 셀은 job을 띄우기만 하고
기다리지 않습니다. 셀이 끝나더라도 학습은 계속 돌아갑니다.

셀이 끝난 다음에 할 일은 두 가지입니다. 먼저 14번 절의 상태 확인 셀을 실행해서 첫 checkpoint가
제대로 저장되는지 확인하십시오. `save_every_steps` 가 50이므로 50 step 뒤에 첫 checkpoint가
생깁니다. 그 확인이 끝나면 이 노트북을 그대로 두고 기다립니다. 15번 절부터는 학습이 끝난 뒤에
실행하는 셀이며, 학습이 도는 중에 실행하면 그 사실을 알아채고 아무 일도 하지 않은 채 안내만
출력합니다.

학습을 중간에 멈춰야 하면 `python -m vqa.jobs stop --name <job 이름>` 을 실행하십시오. 학습
프로세스가 종료 신호를 받아 step 경계에서 checkpoint를 저장한 다음 끝납니다. 다시 시작할
때에는 이 셀을 그대로 다시 실행하면 저장된 지점에서 이어집니다.

아래 셀은 시작하기 전에 두 가지를 확인합니다. 먼저 지금 GPU가 정책이 요구하는 정밀도를
감당하는지 보고, 그다음으로 이 run이 이미 다른 정밀도로 시작된 적이 있는지를
`run_manifest.json` 에서 확인합니다. 정밀도가 다르면 학습을 시작하지 않고 멈춥니다. 같은
run-id 아래에 bf16 checkpoint와 nf4 checkpoint가 섞이면 이어서 하는 실행을 믿을 수 없게 되기
때문입니다.

학습에 쓰는 분할은 파라미터 셀의 `TRAIN_SCOPE` 가 정합니다. `fold0` 이면 5,358문항을,
`full` 이면 6,690문항을 학습하므로 전체 step 수도 그만큼 늘어납니다. 5번 절의 출력에서 학습
범위와 분할 파일을 한 번 더 확인하신 다음에 이 셀을 실행하십시오.

In [11]:
if not TRAINING_ALLOWED:
    raise RuntimeError(
        "현재 GPU의 VRAM이 %.1fGB이므로 bf16 학습을 시작하거나 이어서 할 수 없습니다. "
        "런타임을 끊고 80GB A100으로 다시 연결한 다음 이 셀을 실행하십시오. "
        "지금 이 가상 머신에서는 추론만 nf4로 돌릴 수 있습니다." % VRAM_GB
    )

# 이미 시작한 run이라면 그때 쓴 정밀도와 지금 정밀도가 같아야 합니다.
_manifest_path = RUN_ROOT_P / FT_RUN / "run_manifest.json"
if _manifest_path.exists():
    _manifest = json.loads(_manifest_path.read_text(encoding="utf-8"))
    _prev = str(_manifest.get("config", {}).get("precision", "")).lower()
    if _prev and _prev != PRECISION:
        raise RuntimeError(
            "run %s 은 %s 로 시작했는데 지금 설정은 %s 입니다. 정밀도가 다른 채로 이어서 하면 "
            "checkpoint가 섞입니다. 같은 정밀도의 GPU로 다시 연결하거나, 다른 run-id를 쓰십시오.\n"
            "기록 파일: %s" % (FT_RUN, _prev, PRECISION, _manifest_path)
        )
    print("run %s 의 기존 정밀도 %s 와 현재 정밀도가 같습니다. 이어서 진행합니다." % (FT_RUN, _prev))

cmd = [PY, "-m", "vqa.train",
       "--config", CONFIG, "--run-id", FT_RUN,
       "--split", SPLIT_FILE, "--resume", "auto"] + OVERRIDES
gpu_step(FT_RUN + "_train", cmd, wait=False)

$ /usr/bin/python3 -m vqa.jobs start --name colab27b_ft_bf16_full_v2_read_text_upscale_train --auto-restart 3 --min-free-gb 60 -- /usr/bin/python3 -m vqa.train --config configs/colab_qwen35_27b_bf16_upscale.yaml --run-id colab27b_ft_bf16_full_v2_read_text_upscale --split artifacts/data/split_full.json --resume auto --set prompt_id=v2_read_text
[jobs] 시작: colab27b_ft_bf16_full_v2_read_text_upscale_train pid=12889 log=/content/drive/MyDrive/vqa/runs/jobs/colab27b_ft_bf16_full_v2_read_text_upscale_train.log
[rc=0] 8.0초
job colab27b_ft_bf16_full_v2_read_text_upscale_train 을 띄웠습니다. 이 셀은 기다리지 않고 끝납니다.


'started'

## 14. 학습 상태와 속도를 확인합니다

이 셀은 언제든 다시 실행해도 됩니다. GPU를 쓰지 않으므로 학습이 도는 중에 실행해도 안전합니다.
처음에는 첫 checkpoint가 저장되는지 확인하는 용도로 쓰고, 그 뒤에는 남은 시간을 가늠하는
용도로 씁니다.

출력에서 확인할 것은 세 가지입니다. `heartbeat.json` 의 갱신 시각이 최근인지, step이 실제로
늘고 있는지, 그리고 초당 진행 속도가 preflight에서 잰 값과 비슷한지를 보십시오. 속도가 두 배
가까이 느리다면 가속 커널이 빠진 것이므로 4번 절의 경고를 다시 확인해야 합니다.

In [16]:
WATCH = FT_RUN + "_train"
print("job 상태")
job_status(WATCH)

_log_path = RUN_ROOT_P / FT_RUN / "train_log.jsonl"
_rows = []
if _log_path.exists():
    for _line in _log_path.read_text(encoding="utf-8").splitlines():
        _line = _line.strip()
        if _line:
            try:
                _rows.append(json.loads(_line))
            except ValueError:
                pass

print()
if not _rows:
    print("아직 train_log.jsonl에 기록이 없습니다. 모델을 올리는 중일 수 있습니다.")
    print("몇 분 뒤에 이 셀을 다시 실행하십시오. 로그는 다음 명령으로도 볼 수 있습니다.")
    print("  python -m vqa.jobs tail --name %s --lines 40" % WATCH)
else:
    _recent = _rows[-8:]
    print("최근 기록 %d줄" % len(_recent))
    print("  %8s %10s %12s %12s" % ("step", "loss", "초/step", "peak VRAM"))
    for _row in _recent:
        print("  %8s %10.4f %12.2f %12s"
              % (_row.get("step"), float(_row.get("loss") or 0.0),
                 float(_row.get("seconds_per_step") or 0.0), _row.get("peak_vram_gb")))

    _last = _rows[-1]
    _speeds = [float(r["seconds_per_step"]) for r in _recent if r.get("seconds_per_step")]
    _mean_speed = sum(_speeds) / len(_speeds) if _speeds else 0.0
    _step = int(_last.get("step") or 0)
    _total = int(_last.get("total_steps") or 0)
    _remaining = max(_total - _step, 0)
    print()
    print("진행            : %d / %d step (%.1f%%)"
          % (_step, _total, (100.0 * _step / _total) if _total else 0.0))
    print("최근 평균 속도  : %.2f 초/optimizer step" % _mean_speed)
    print("남은 step       : %d" % _remaining)
    if _mean_speed:
        _eta = _remaining * _mean_speed
        print("예상 남은 시간  : %.1f시간 (%.0f분)" % (_eta / 3600.0, _eta / 60.0))
    print("peak VRAM       : %s GB" % _last.get("peak_vram_gb"))

_hb_path = RUN_ROOT_P / FT_RUN / "heartbeat.json"
if _hb_path.exists():
    _hb = json.loads(_hb_path.read_text(encoding="utf-8"))
    print()
    print("heartbeat 갱신  : %s" % _hb.get("updated"))
    print("heartbeat ETA   : %s" % _hb.get("eta_human"))

_latest = RUN_ROOT_P / FT_RUN / "checkpoints" / "latest.json"
print()
if _latest.exists():
    print("마지막 checkpoint: %s" % _latest.read_text(encoding="utf-8").strip()[:300])
else:
    print("아직 checkpoint가 없습니다. save_every_steps 만큼 진행해야 첫 checkpoint가 생깁니다.")

job 상태
job colab27b_ft_bf16_full_v2_read_text_upscale_train: DEAD pid=12889
  child   : pid=None DEAD restarts=0
  started : 2026-09-22T10:02:49+00:00
  command : /usr/bin/python3 -m vqa.train --config configs/colab_qwen35_27b_bf16_upscale.yaml --run-id colab27b_ft_bf16_full_v2_read_text_upscale --split artifacts/data/split_full.json --resume auto --set prompt_id=v2_read_text
  log     : /content/drive/MyDrive/vqa/runs/jobs/colab27b_ft_bf16_full_v2_read_text_upscale_train.log
  exit    : 0 (2026-09-22T13:32:12+00:00)
  progress: step 837/837 loss_ema=0.08181453150284414 s/step=14.389350562095641 ETA=0s
  vram    : peak 59.58744668960571 GB, heartbeat 3966.501918초 전
  ckpt    : step 837 (step_000837, 2026-09-22T13:31:26+00:00)

최근 기록 8줄
      step       loss       초/step    peak VRAM
       830     0.0012        14.60 59.58744668960571
       831     0.0037        14.59 59.58744668960571
       832     0.0023        14.58 59.58744668960571
       833     0.0059        14.59 59.587446689

In [13]:
run([PY, "-m", "vqa.jobs", "wait", "--name", FT_RUN + "_train", "--poll", "60"], check=False)

$ /usr/bin/python3 -m vqa.jobs wait --name colab27b_ft_bf16_full_v2_read_text_upscale_train --poll 60
[jobs] colab27b_ft_bf16_full_v2_read_text_upscale_train 대기 중 pid=12889 child_pid=12891 | [jobs] supervise 시작 attempt=0: /usr/bin/python3 -m vqa.train --config configs/colab_qwen35_27b_bf16_upscale.yaml --run-id colab27b_ft_bf16_full_v2_read_text_upscale --split artifacts/data/split_full.json --resume auto --set prompt_id=v2_read_text
[jobs] colab27b_ft_bf16_full_v2_read_text_upscale_train 대기 중 pid=12889 child_pid=12891 | Loading weights:   0%|          | 0/1184 [00:00<?, ?it/s]
[jobs] colab27b_ft_bf16_full_v2_read_text_upscale_train 대기 중 pid=12889 child_pid=12891 | Loading weights:  15%|█▌        | 182/1184 [01:23<04:06,  4.07it/s]
[jobs] colab27b_ft_bf16_full_v2_read_text_upscale_train 대기 중 pid=12889 child_pid=12891 | Loading weights:  31%|███▏      | 370/1184 [02:24<04:23,  3.09it/s]
[jobs] colab27b_ft_bf16_full_v2_read_text_upscale_train 대기 중 pid=12889 child_pid=12891 | Loading weig

{'rc': 0,
 'seconds': 12606.468311309814,
 'output': '[jobs] colab27b_ft_bf16_full_v2_read_text_upscale_train 대기 중 pid=12889 child_pid=12891 | [jobs] supervise 시작 attempt=0: /usr/bin/python3 -m vqa.train --config configs/colab_qwen35_27b_bf16_upscale.yaml --run-id colab27b_ft_bf16_full_v2_read_text_upscale --split artifacts/data/split_full.json --resume auto --set prompt_id=v2_read_text\n[jobs] colab27b_ft_bf16_full_v2_read_text_upscale_train 대기 중 pid=12889 child_pid=12891 | Loading weights:   0%|          | 0/1184 [00:00<?, ?it/s]\n[jobs] colab27b_ft_bf16_full_v2_read_text_upscale_train 대기 중 pid=12889 child_pid=12891 | Loading weights:  15%|█▌        | 182/1184 [01:23<04:06,  4.07it/s]\n[jobs] colab27b_ft_bf16_full_v2_read_text_upscale_train 대기 중 pid=12889 child_pid=12891 | Loading weights:  31%|███▏      | 370/1184 [02:24<04:23,  3.09it/s]\n[jobs] colab27b_ft_bf16_full_v2_read_text_upscale_train 대기 중 pid=12889 child_pid=12891 | Loading weights: 100%|██████████| 1184/1184 [03:15<00:00

## 15. 학습이 끝난 뒤: fold 0을 다시 추론하고 평가합니다

여기부터는 학습이 끝난 다음에 실행하는 셀입니다. 14번 절의 상태가 완료로 바뀌었는지 먼저
확인하십시오. 15번 절부터 17번 절까지의 셀은 실행하기 전에 `post_train_ready` 로 학습 job이
아직 살아 있는지와 `final_adapter` 가 만들어졌는지를 확인합니다. 둘 중 하나라도 충족되지
않으면 안내만 출력하고 아무 일도 하지 않습니다. 그래서 메뉴의 "모두 실행"으로 노트북 전체를
돌려도 학습과 추론이 GPU를 함께 쓰는 일이 생기지 않습니다. 학습이 끝난 뒤에 이 아래의 셀들을
차례로 다시 실행하시면 됩니다.

파라미터 셀의 `ALSO_BASE` 가 `True` 이면 predict 명령에 `--also-base` 가 붙어서, 같은
forward에서 원본 모델의 확률도 함께 남습니다. 결합 후보를 비교할 때 GPU 추론을 한 번 더 하지
않고 CPU에서만 계산할 수 있게 됩니다. 다만 기본값은 `False` 입니다. 같은 프롬프트의 원본 예측을
9번 절에서 이미 만들어 두었다면 같은 값을 다시 계산하게 되므로, 추론 시간만 두 배 가까이
늘어나기 때문입니다.

`TRAIN_SCOPE` 가 `full` 이면 이 절의 두 셀은 안내만 출력하고 건너뜁니다. 전체 데이터로 학습한
모델은 fold 0의 1,344문항도 학습에 썼으므로, 그 문항을 다시 맞힌 정확도로는 일반화 성능도 학습
효과도 판단할 수 없습니다.

In [ ]:
if fold0_eval_skipped():
    pass
elif post_train_ready():
    cmd = [PY, "-m", "vqa.predict",
           "--config", CONFIG, "--run-id", FT_RUN,
           "--csv", "train.csv", "--ids-file", VAL_IDS_FILE,
           "--adapter", ADAPTER, "--out-name", VAL_OUT_FT] + ALSO_BASE_ARGS + OVERRIDES
    gpu_step(FT_RUN + "_val" + XADAPTER_SUFFIX, cmd, done=pred_done(FT_RUN, VAL_OUT_FT))

In [ ]:
if fold0_eval_skipped():
    pass
elif not pred_done(FT_RUN, VAL_OUT_FT):
    print("학습된 모델의 fold 0 예측이 아직 끝나지 않아 평가를 건너뜁니다.")
    print("바로 위 셀이 끝난 다음에 이 셀을 다시 실행하십시오.")
else:
    evaluate(pred_file(FT_RUN, VAL_OUT_FT), FT_RUN + "_" + VAL_OUT_FT,
             "27B LoRA, 프롬프트 %s, 정밀도 %s, adapter %s"
             % (PROMPT_ID, PRECISION, ADAPTER_SOURCE_RUN))

## 16. 학습 전과 학습 후를 문항 단위로 비교합니다

정확도 두 개를 나란히 놓고 보는 것만으로는 학습이 도움이 되었는지 판단할 수 없습니다. fold 0은
1,344문항뿐이라서 0.5퍼센트포인트 정도의 차이는 표본 변동만으로도 생깁니다. 두 모델이 같은
문항을 풀었으므로 짝지은 비교를 해야 하고, 그 자리에 맞는 검정이 McNemar 검정입니다.

아래 셀은 한쪽만 맞힌 문항 수 두 개를 세서 양측 정확 이항 검정의 p값을 계산합니다. p값이
0.05보다 크면 두 모델의 차이를 표본 변동과 구분할 수 없다는 뜻이므로, 정확도가 조금 올랐더라도
학습이 도움이 되었다고 팀 채널에 보고하지 마십시오. 어떤 문항이 달라졌는지는
`artifacts/eval/base_vs_ft_changed.csv` 에 저장됩니다.

이 비교는 `TRAIN_SCOPE` 가 `fold0` 일 때에만 뜻이 있습니다. `full` 이면 학습 모델이 fold 0을
이미 학습에 썼으므로 짝지은 비교가 성립하지 않고, 아래 셀은 안내만 출력하고 건너뜁니다. 학습
설정이 도움이 되는지에 대한 판단은 `fold0` 으로 돌린 run에서 이미 끝났다고 보시면 됩니다.

In [ ]:
if fold0_eval_skipped():
    pass
elif not (pred_done(BASE_RUN, VAL_OUT_BASE) and pred_done(FT_RUN, VAL_OUT_FT)):
    print("비교에 필요한 예측 두 개가 아직 모두 끝나지 않아 이 셀은 건너뜁니다.")
    print("  원본 모델 fold 0 : %s" % ("완료" if pred_done(BASE_RUN, VAL_OUT_BASE) else "미완료"))
    print("  학습 모델 fold 0 : %s" % ("완료" if pred_done(FT_RUN, VAL_OUT_FT) else "미완료"))
else:
    Path("artifacts/eval").mkdir(parents=True, exist_ok=True)
    script("compare_predictions.py",
           "--pred-a", str(pred_file(BASE_RUN, VAL_OUT_BASE)),
           "--pred-b", str(pred_file(FT_RUN, VAL_OUT_FT)),
           "--gold", str(DATA_ROOT_P / "train.csv"),
           "--val-ids", VAL_IDS_FILE,
           "--label-a", "base", "--label-b", "ft",
           "--changed-out", "artifacts/eval/base_vs_ft%s_changed.csv" % XADAPTER_SUFFIX,
           "--json-out", "artifacts/eval/base_vs_ft%s.json" % XADAPTER_SUFFIX,
           check=False)

## 17. 학습된 모델로 전체 test를 추론하고 제출 파일을 만듭니다

16번 절에서 학습이 도움이 되었다고 판단한 경우에만 이 단계를 진행하십시오. 도움이 되지
않았다면 12번 절에서 만든 원본 모델의 제출 파일이 더 나은 후보입니다.

In [14]:
if post_train_ready():
    cmd = [PY, "-m", "vqa.predict",
           "--config", CONFIG, "--run-id", FT_RUN,
           "--csv", "test.csv",
           "--adapter", ADAPTER, "--out-name", TEST_OUT_FT] + ALSO_BASE_ARGS + OVERRIDES
    gpu_step(FT_RUN + "_test" + XADAPTER_SUFFIX, cmd, done=pred_done(FT_RUN, TEST_OUT_FT))

$ /usr/bin/python3 -m vqa.jobs start --name colab27b_ft_bf16_full_v2_read_text_upscale_test --auto-restart 3 --min-free-gb 60 -- /usr/bin/python3 -m vqa.predict --config configs/colab_qwen35_27b_bf16_upscale.yaml --run-id colab27b_ft_bf16_full_v2_read_text_upscale --csv test.csv --adapter /content/drive/MyDrive/vqa/runs/colab27b_ft_bf16_full_v2_read_text_upscale/final_adapter --out-name test_ft --set prompt_id=v2_read_text
[jobs] 시작: colab27b_ft_bf16_full_v2_read_text_upscale_test pid=68258 log=/content/drive/MyDrive/vqa/runs/jobs/colab27b_ft_bf16_full_v2_read_text_upscale_test.log
[rc=0] 9.0초
job colab27b_ft_bf16_full_v2_read_text_upscale_test 이 끝날 때까지 기다립니다. 이 셀을 멈춰도 job은 계속 돌아갑니다.
$ /usr/bin/python3 -m vqa.jobs wait --name colab27b_ft_bf16_full_v2_read_text_upscale_test --poll 30
[jobs] colab27b_ft_bf16_full_v2_read_text_upscale_test 대기 중 pid=68258 child_pid=68260 | [jobs] supervise 시작 attempt=0: /usr/bin/python3 -m vqa.predict --config configs/colab_qwen35_27b_bf16_upscale.yaml --r

In [15]:
if not pred_done(FT_RUN, TEST_OUT_FT):
    print("학습된 모델의 test 예측이 아직 끝나지 않아 제출 파일을 만들지 않습니다.")
    print("바로 위 셀이 끝난 다음에 이 셀을 다시 실행하십시오.")
else:
    make_submission(pred_file(FT_RUN, TEST_OUT_FT),
                    "artifacts/submission/%s_ft%s.csv" % (FT_RUN, XADAPTER_SUFFIX))

$ /usr/bin/python3 -m vqa.submit --pred /content/drive/MyDrive/vqa/runs/colab27b_ft_bf16_full_v2_read_text_upscale/preds/test_ft.csv --sample /content/data/sample_submission.csv --out artifacts/submission/colab27b_ft_bf16_full_v2_read_text_upscale_ft.csv
제출 파일 생성  : artifacts/submission/colab27b_ft_bf16_full_v2_read_text_upscale_ft.csv (6714행)

파일        : artifacts/submission/colab27b_ft_bf16_full_v2_read_text_upscale_ft.csv
sha256      : c9fd8b106b9c28bd7c7f9d0602be3edf171e8c1d7018acd1b17647de19b905d3
행 수       : 6714
answer 분포 :
  a:   1717  (25.57%)
  b:   1672  (24.90%)
  c:   1640  (24.43%)
  d:   1685  (25.10%)

결과: 통과
[rc=0] 2.8초
$ /usr/bin/python3 scripts/check_submission.py --submission artifacts/submission/colab27b_ft_bf16_full_v2_read_text_upscale_ft.csv --sample /content/data/sample_submission.csv
파일        : artifacts/submission/colab27b_ft_bf16_full_v2_read_text_upscale_ft.csv
sha256      : c9fd8b106b9c28bd7c7f9d0602be3edf171e8c1d7018acd1b17647de19b905d3
행 수       : 6714

In [17]:
!mkdir -p /content/drive/MyDrive/vqa/artifacts_20260923_full_upscale
!cp -r /content/ssafy-16-2-ai/artifacts/eval /content/ssafy-16-2-ai/artifacts/submission /content/ssafy-16-2-ai/artifacts/experiments.jsonl /content/drive/MyDrive/vqa/artifacts_20260923_full_upscale/
!ls -l /content/drive/MyDrive/vqa/artifacts_20260923_full_upscale/submission

cp: cannot stat '/content/ssafy-16-2-ai/artifacts/eval': No such file or directory
cp: cannot stat '/content/ssafy-16-2-ai/artifacts/experiments.jsonl': No such file or directory
total 105
-rw------- 1 root root 107434 Sep 22 14:38 colab27b_ft_bf16_full_v2_read_text_upscale_ft.csv


## 18. 실행 상태를 팀에 공유합니다

아래 출력을 팀 채널에 붙여 넣습니다. 다른 팀원은 실행 중인 런타임을 볼 수 없으므로, 이 기록이
유일한 공유 수단입니다.

In [ ]:
JOB_NAMES = ["preflight_" + PRECISION,
             BASE_RUN + "_val", BASE_RUN + "_test",
             FT_RUN + "_train",
             FT_RUN + "_val" + XADAPTER_SUFFIX,
             FT_RUN + "_test" + XADAPTER_SUFFIX]
if RUN_PROMPT_COMPARE:
    JOB_NAMES.append(CMP_RUN + "_val")
if RUN_RESOLUTION_COMPARE:
    JOB_NAMES.append(RES_CMP_RUN + "_val")

lines = ["===== Colab 27B 실행 상태 ====="]
lines.append("시각      : %s" % time.strftime("%Y-%m-%d %H:%M:%S"))
lines.append("GPU       : %s / VRAM %.1fGB" % (GPU_NAME, VRAM_GB))
lines.append("config    : %s" % CONFIG)
lines.append("프롬프트  : %s" % PROMPT_ID)
lines.append("코드 버전 : %s" % (CODE_VERSION or "(알 수 없음)"))
for name in JOB_NAMES:
    res = run([PY, "-m", "vqa.jobs", "status", "--name", name],
              check=False, quiet=True, echo=False, hint=False)
    first = res["output"].strip().splitlines()
    lines.append("%-30s %s" % (name, first[0] if first else "기록 없음"))
REPORT_RUNS = [BASE_RUN, FT_RUN]
if RUN_RESOLUTION_COMPARE:
    REPORT_RUNS.append(RES_CMP_RUN)
for run_id in REPORT_RUNS:
    pred_dir = RUN_ROOT_P / run_id / "preds"
    if pred_dir.is_dir():
        for f in sorted(pred_dir.glob("*.csv")):
            done = "완료" if Path(str(f) + ".manifest.json").exists() else "진행 중"
            lines.append("예측 파일 : %s/%s (%d바이트, %s)" % (run_id, f.name, f.stat().st_size, done))
lines.append("==============================")
print("\n".join(lines))

## 부록 A. 백그라운드 실행 확인 (D1에 한 번만)

Pro+의 백그라운드 실행이 이 계정에서 실제로 어떻게 동작하는지 직접 확인하는 절차입니다.
안내 문구만 믿고 밤샘 실행을 맡기면 안 되므로, 한 번은 직접 재 보아야 합니다. 본 실행 흐름과는
분리되어 있으므로 필요할 때에만 실행하십시오.

절차는 다음과 같습니다.

1. 아래 셀을 실행해서 15분짜리 시험용 job을 띄웁니다. 이 job은 1분마다 Drive의 파일에 현재
   시각을 기록합니다.
2. 브라우저 탭을 완전히 닫습니다. 창을 최소화하거나 다른 탭으로 옮기는 것이 아니라 탭 자체를
   닫아야 합니다.
3. 10분 정도 지난 뒤에 다시 접속합니다.
4. 확인 셀을 실행해서 기록된 시각 사이에 빈 구간이 있는지 봅니다.

판정 기준은 다음과 같습니다. 기록이 1분 간격으로 끊김 없이 이어지면 탭을 닫아도 실행이 유지되는
것입니다. 탭을 닫은 시점부터 기록이 멈췄다가 다시 접속한 시점부터 다시 시작되면, 백그라운드
실행이 동작하지 않는 것이므로 야간 실행 계획을 다시 세워야 합니다. 확인 결과는 팀 채널에 반드시
공유하십시오.

이 시험용 job은 GPU를 쓰지 않습니다. 다만 `vqa.jobs start` 는 같은 run-root에 살아 있는 job이
있으면 시작을 거부하므로, 다른 작업이 없을 때 실행하십시오.

In [ ]:
HEARTBEAT_FILE = RUN_ROOT_P / "bg_test_heartbeat.txt"
SCRIPT_PATH = RUN_ROOT_P / "bg_test.py"
SCRIPT_PATH.write_text(
    "import time, datetime, sys, pathlib\n"
    "out = pathlib.Path(sys.argv[1])\n"
    "for i in range(15):\n"
    "    with out.open('a', encoding='utf-8') as fh:\n"
    "        fh.write(datetime.datetime.now().isoformat(timespec='seconds') + '\\n')\n"
    "        fh.flush()\n"
    "    time.sleep(60)\n"
    "print('시험용 job을 마쳤습니다.')\n",
    encoding="utf-8",
)
if HEARTBEAT_FILE.exists():
    HEARTBEAT_FILE.unlink()

gpu_step("bg_test", [PY, str(SCRIPT_PATH), str(HEARTBEAT_FILE)],
         wait=False, min_free_gb=0)
print()
print("지금부터 브라우저 탭을 닫고 10분쯤 뒤에 다시 접속하십시오.")
print("기록 파일:", HEARTBEAT_FILE)

In [ ]:
import datetime

if not HEARTBEAT_FILE.exists():
    print("아직 기록이 없습니다. job이 시작되었는지 status로 확인하십시오.")
else:
    stamps = [datetime.datetime.fromisoformat(s)
              for s in HEARTBEAT_FILE.read_text(encoding="utf-8").split() if s]
    print("기록 %d개" % len(stamps))
    gaps = []
    for a, b in zip(stamps, stamps[1:]):
        sec = (b - a).total_seconds()
        gaps.append(sec)
        flag = "  <-- 끊김" if sec > 120 else ""
        print("  %s -> %s  %5.0f초%s" % (a.time(), b.time(), sec, flag))
    if gaps and max(gaps) <= 120:
        print()
        print("판정: 탭을 닫아도 실행이 유지되었습니다. 야간 실행을 맡길 수 있습니다.")
    elif gaps:
        print()
        print("판정: 끊긴 구간이 있습니다. 백그라운드 실행을 믿지 말고,")
        print("      checkpoint 간격을 짧게 잡고 재개 절차를 자주 확인해야 합니다.")